In [1]:
import os
from pyspark.sql import SparkSession, functions as F

In [2]:
CATALOG = "lakehouse"
C = f"spark.sql.catalog.{CATALOG}"

spark = (
    SparkSession.builder
    .appName("Lakehouse Implementation")
    .master("spark://spark-master:7077")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")

    # Named Iceberg catalog backed by the Hive Metastore
    .config(C, "org.apache.iceberg.spark.SparkCatalog")
    .config(f"{C}.type", "hive")
    .config(f"{C}.uri", "thrift://hive-metastore:9083")
    .config(f"{C}.warehouse", "s3://buck1/warehouse")

    # Iceberg's own S3 client for table data and metadata.
    # Keys and region come from AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY / AWS_REGION.
    .config(f"{C}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"{C}.s3.endpoint", os.environ["S3_ENDPOINT"])
    .config(f"{C}.s3.path-style-access", "true")
    .config(f"{C}.http-client.type", "apache")   # the only HTTP client in bundle-2.29.52.jar

    .config("spark.sql.defaultCatalog", CATALOG)

    # S3A is still used for streaming checkpoints (Spark writes them through Hadoop, not Iceberg).
    # S3A's default credential chain reads the same AWS_* environment variables.
    .config("spark.hadoop.fs.s3a.endpoint", os.environ["S3_ENDPOINT"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    # FORMAT_LAB (Stage 5+): Hive tables in spark_catalog, stored in the same Hive Metastore
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")
    .enableHiveSupport()

    .getOrCreate()
)

BASE = "s3a://format-lab"
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/23 04:19:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Helper: `tree(path)`
Lists the files under a path with their sizes, using Spark's own Hadoop FileSystem (no boto3 needed yet).

In [3]:
def tree(path):
    """Print every file under `path` with its size in KB, and return the total bytes."""
    jvm = spark._jvm
    p = jvm.org.apache.hadoop.fs.Path(path)
    fs = p.getFileSystem(spark._jsc.hadoopConfiguration())
    files = fs.listFiles(p, True)          # True = recursive
    total = 0
    while files.hasNext():
        f = files.next()
        rel = f.getPath().toString().replace(path, "")
        # print(f"{f.getLen()/1024:10.1f} KB  {rel}")
        total += f.getLen()
    # print(f"{total/1024:10.1f} KB  TOTAL")
    return total

### Test dataset (~10k rows)
Flat columns only for now (CSV can't hold struct/array). Every 10th row has nulls.

In [4]:
df = (
    spark.range(10_000)
    .select(
        F.col("id").cast("int").alias("id"),
        (F.col("id") * 1_000_000).alias("big_id"),                                 # bigint
        F.concat(F.lit("customer_"), F.col("id") % 500).alias("customer"),         # string
        (F.rand(42) * 1000).cast("decimal(10,2)").alias("amount"),                # decimal
        F.date_add(F.lit("2026-01-01"), (F.col("id") % 5).cast("int")).alias("sale_date"),  # date
        F.timestamp_seconds(F.lit(1767225600) + F.col("id") * 60).alias("sale_ts"),         # timestamp
        (F.col("id") % 2 == 0).alias("is_paid"),                                   # boolean
    )
    # put nulls in some rows
    .withColumn("customer", F.when(F.col("id") % 10 == 0, None).otherwise(F.col("customer")))
    .withColumn("amount",   F.when(F.col("id") % 10 == 0, None).otherwise(F.col("amount")))
)

df.printSchema()
df.show(5, truncate=False)

root
 |-- id: integer (nullable = false)
 |-- big_id: long (nullable = false)
 |-- customer: string (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- sale_ts: timestamp (nullable = false)
 |-- is_paid: boolean (nullable = true)



+---+-------+----------+------+----------+-------------------+-------+
|id |big_id |customer  |amount|sale_date |sale_ts            |is_paid|
+---+-------+----------+------+----------+-------------------+-------+
|0  |0      |NULL      |NULL  |2026-01-01|2026-01-01 00:00:00|true   |
|1  |1000000|customer_1|509.60|2026-01-02|2026-01-01 00:01:00|false  |
|2  |2000000|customer_2|832.53|2026-01-03|2026-01-01 00:02:00|true   |
|3  |3000000|customer_3|263.23|2026-01-04|2026-01-01 00:03:00|false  |
|4  |4000000|customer_4|670.29|2026-01-05|2026-01-01 00:04:00|true   |
+---+-------+----------+------+----------+-------------------+-------+
only showing top 5 rows


# Stage 1: Text files (CSV, JSON)
*Starting point: data stored as human-readable text.*

In [5]:
TEXT = f"{BASE}/files/text"

In [12]:
# 1.1 Write CSV (with and without header) and JSON (one object per line)
# `coalesce(1)` = one file, so it's easy to look at in MinIO.
df.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{TEXT}/csv_header")
df.coalesce(1).write.mode("overwrite").option("header", False).csv(f"{TEXT}/csv_no_header")
df.coalesce(1).write.mode("overwrite").json(f"{TEXT}/json")

# tree(TEXT)

26/11/20 05:03:53 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/11/20 05:03:56 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/11/20 05:03:56 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


In [16]:
# 1.2 Look at the raw file content
# Plain text: readable, but there are no types and no schema in the file.
for name in ["csv_header", "csv_no_header", "json"]:
    print(f"--- {name} ---")
    spark.read.text(f"{TEXT}/{name}").show(3, truncate=False)

--- csv_header ---
+---------------------------------------------------------------------+
|value                                                                |
+---------------------------------------------------------------------+
|id,big_id,customer,amount,sale_date,sale_ts,is_paid                  |
|0,0,,,2026-01-01,2026-01-01T00:00:00.000Z,true                       |
|1,1000000,customer_1,509.60,2026-01-02,2026-01-01T00:01:00.000Z,false|
+---------------------------------------------------------------------+
only showing top 3 rows
--- csv_no_header ---


+---------------------------------------------------------------------+
|value                                                                |
+---------------------------------------------------------------------+
|0,0,,,2026-01-01,2026-01-01T00:00:00.000Z,true                       |
|1,1000000,customer_1,509.60,2026-01-02,2026-01-01T00:01:00.000Z,false|
|2,2000000,customer_2,832.53,2026-01-03,2026-01-01T00:02:00.000Z,true |
+---------------------------------------------------------------------+
only showing top 3 rows
--- json ---
+-----------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                          |
+-----------------------------------------------------------------------------------------------------------------------------------------------+
|{"id":0,"big_id":0,"

In [17]:
# 1.3 Read back without a schema
# Compare with the original schema above: what got the wrong type?
print("ORIGINAL"); df.printSchema()

print("CSV header, no inferSchema  -> everything is string")
spark.read.option("header", True).csv(f"{TEXT}/csv_header").printSchema()

print("CSV header, inferSchema=True -> Spark guesses from the values")
spark.read.option("header", True).option("inferSchema", True).csv(f"{TEXT}/csv_header").printSchema()

print("CSV no header -> column names are lost (_c0, _c1, ...)")
spark.read.csv(f"{TEXT}/csv_no_header").printSchema()

print("JSON -> types guessed from JSON values")
spark.read.json(f"{TEXT}/json").printSchema()

ORIGINAL
root
 |-- id: integer (nullable = false)
 |-- big_id: long (nullable = false)
 |-- customer: string (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- sale_ts: timestamp (nullable = false)
 |-- is_paid: boolean (nullable = true)

CSV header, no inferSchema  -> everything is string
root
 |-- id: string (nullable = true)
 |-- big_id: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- sale_ts: string (nullable = true)
 |-- is_paid: string (nullable = true)

CSV header, inferSchema=True -> Spark guesses from the values


root
 |-- id: integer (nullable = true)
 |-- big_id: long (nullable = true)
 |-- customer: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- sale_ts: timestamp (nullable = true)
 |-- is_paid: boolean (nullable = true)

CSV no header -> column names are lost (_c0, _c1, ...)
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)

JSON -> types guessed from JSON values
root
 |-- amount: double (nullable = true)
 |-- big_id: long (nullable = true)
 |-- customer: string (nullable = true)
 |-- id: long (nullable = true)
 |-- is_paid: boolean (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- sale_ts: string (nullable = true)



**What to notice**
- `decimal(10,2)` → `double` (precision lost)
- `date` → `string` or `date` depending on inference; JSON gives `string`
- JSON column order is alphabetical, not the original order
- JSON **drops null fields** from the line (open the file and check rows with id 0, 10, 20...)
- Without a header, column names are gone

In [21]:
# 1.4 Compression: none vs gzip
df.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{TEXT}/csv_plain")
df.coalesce(1).write.mode("overwrite").option("header", True).option("compression", "gzip").csv(f"{TEXT}/csv_gzip")
df.coalesce(1).write.mode("overwrite").option("compression", "gzip").json(f"{TEXT}/json_gzip")

sizes = {name: tree(f"{TEXT}/{name}") for name in ["csv_plain", "csv_gzip", "json", "json_gzip"]}
for name, size in sizes.items():
    print(f"{name:10} {size/1024:8.1f} KB")

26/11/20 05:25:54 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/11/20 05:25:54 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/11/20 05:25:55 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


csv_plain     734.2 KB
csv_gzip      138.2 KB
json         1433.4 KB
json_gzip     156.0 KB


In [22]:
# 1.5 Splittable or not?
# A plain CSV can be cut into pieces and read in parallel. A `.gz` file can't: one task must read it from the start.
# We set a small max split size so the difference is visible.

spark.conf.set("spark.sql.files.maxPartitionBytes", 64 * 1024)   # 64 KB splits

plain = spark.read.option("header", True).csv(f"{TEXT}/csv_plain")
gz    = spark.read.option("header", True).csv(f"{TEXT}/csv_gzip")

print("plain CSV partitions:", plain.rdd.getNumPartitions())   # several
print("gzip  CSV partitions:", gz.rdd.getNumPartitions())      # 1

spark.conf.unset("spark.sql.files.maxPartitionBytes")

plain CSV partitions: 12
gzip  CSV partitions: 1


**Lesson → Stage 2:** no schema, no types, large files, slow parsing.

# Stage 2: Avro (binary, row-based)
*Fixes: no schema, no types. The schema is embedded in every file.*

In [5]:
AVRO = f"{BASE}/files/avro"

In [9]:
# helper: copy the first file with a given ending from MinIO to the container's local disk,
# so Python libraries (fastavro, pyarrow) can open it
def download_first(path, suffix, local):
    jvm = spark._jvm
    p = jvm.org.apache.hadoop.fs.Path(path)
    fs = p.getFileSystem(spark._jsc.hadoopConfiguration())
    for st in fs.listStatus(p):
        if st.getPath().getName().endswith(suffix):
            fs.copyToLocalFile(st.getPath(), jvm.org.apache.hadoop.fs.Path(local))
            return local

In [7]:
# 2.1 Write Avro and read it back
# No inferSchema needed: the types come back from the file itself.
df.coalesce(1).write.mode("overwrite").format("avro").save(f"{AVRO}/data")

print("ORIGINAL"); df.printSchema()
print("AVRO read back"); spark.read.format("avro").load(f"{AVRO}/data").printSchema()

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/11/20 07:27:34 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


ORIGINAL
root
 |-- id: integer (nullable = false)
 |-- big_id: long (nullable = false)
 |-- customer: string (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- sale_ts: timestamp (nullable = false)
 |-- is_paid: boolean (nullable = true)

AVRO read back
root
 |-- id: integer (nullable = true)
 |-- big_id: long (nullable = true)
 |-- customer: string (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- sale_ts: timestamp (nullable = true)
 |-- is_paid: boolean (nullable = true)



In [9]:
# 2.2 Look inside the file header with fastavro
# Every Avro file starts with a header that holds the full schema as JSON.
import json, fastavro

local = download_first(f"{AVRO}/data", ".avro", "/tmp/sample.avro")

with open(local, "rb") as f:
    reader = fastavro.reader(f)
    print(json.dumps(reader.writer_schema, indent=2))
    print("codec:    ", reader.codec)
    print("first row:", next(reader))

{
  "type": "record",
  "name": "topLevelRecord",
  "fields": [
    {
      "name": "id",
      "type": "int"
    },
    {
      "name": "big_id",
      "type": "long"
    },
    {
      "name": "customer",
      "type": [
        "string",
        "null"
      ]
    },
    {
      "name": "amount",
      "type": [
        {
          "logicalType": "decimal",
          "precision": 10,
          "scale": 2,
          "type": "fixed",
          "name": "topLevelRecord.amount.fixed",
          "size": 5
        },
        "null"
      ]
    },
    {
      "name": "sale_date",
      "type": [
        {
          "logicalType": "date",
          "type": "int"
        },
        "null"
      ]
    },
    {
      "name": "sale_ts",
      "type": {
        "logicalType": "timestamp-micros",
        "type": "long"
      }
    },
    {
      "name": "is_paid",
      "type": [
        "boolean",
        "null"
      ]
    }
  ]
}
codec:     snappy


**What to notice**
- Each field has a real type: `decimal` keeps precision and scale, `date` and `timestamp-micros` are logical types
- Nullable columns are a union: `["string", "null"]`
- Open the file in MinIO: the header JSON is readable, the rows after it are binary

In [10]:
# 2.3 Schema evolution: writer schema vs reader schema
# The v1 file has no `channel` field. Reading it with the v2 schema, which gives `channel`
# a default, fills the default in. So old and new files can be read together.
v1 = {"type": "record", "name": "Sale", "fields": [
    {"name": "id",     "type": "int"},
    {"name": "amount", "type": "double"},
]}
v2 = {"type": "record", "name": "Sale", "fields": [
    {"name": "id",      "type": "int"},
    {"name": "amount",  "type": "double"},
    {"name": "channel", "type": "string", "default": "unknown"},   # new column with a default
]}

with open("/tmp/sales_v1.avro", "wb") as f:
    fastavro.writer(f, v1, [{"id": 1, "amount": 10.5}, {"id": 2, "amount": 20.0}])
with open("/tmp/sales_v2.avro", "wb") as f:
    fastavro.writer(f, v2, [{"id": 3, "amount": 5.0, "channel": "web"}])

for p in ["/tmp/sales_v1.avro", "/tmp/sales_v2.avro"]:
    with open(p, "rb") as f:
        for row in fastavro.reader(f, reader_schema=v2):
            print(p, row)

/tmp/sales_v1.avro {'id': 1, 'amount': 10.5, 'channel': 'unknown'}
/tmp/sales_v1.avro {'id': 2, 'amount': 20.0, 'channel': 'unknown'}
/tmp/sales_v2.avro {'id': 3, 'amount': 5.0, 'channel': 'web'}


In [11]:
# 2.4 Same idea in Spark: give the newer schema when reading old files
# avroSchema = the reader schema; the missing `channel` column gets its default.
# Spark names its Avro record "topLevelRecord", so the reader schema uses the same name.
reader_schema = json.dumps({"type": "record", "name": "topLevelRecord", "fields": [
    {"name": "id",      "type": "int"},
    {"name": "channel", "type": "string", "default": "unknown"},
]})

df.select("id").limit(3).coalesce(1).write.mode("overwrite").format("avro").save(f"{AVRO}/v1_only_id")

spark.read.format("avro").option("avroSchema", reader_schema).load(f"{AVRO}/v1_only_id").show()

26/11/20 07:42:26 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


+----+-------+
|  id|channel|
+----+-------+
|5000|unknown|
|5001|unknown|
|5002|unknown|
+----+-------+



In [14]:
# 2.5 Size compared with CSV / JSON
df.coalesce(1).write.mode("overwrite").format("avro").option("compression", "uncompressed").save(f"{AVRO}/uncompressed")
df.coalesce(1).write.mode("overwrite").format("avro").option("compression", "snappy").save(f"{AVRO}/snappy")

sizes = {
    "csv":           tree(f"{TEXT}/csv_plain"),
    "csv.gz":        tree(f"{TEXT}/csv_gzip"),
    "json":          tree(f"{TEXT}/json"),
    "json.gz":       tree(f"{TEXT}/json_gzip"),
    "avro (none)":   tree(f"{AVRO}/uncompressed"),
    "avro (snappy)": tree(f"{AVRO}/snappy"),
}
for name, size in sizes.items():
    print(f"{name:14} {size/1024:8.1f} KB")

26/11/20 07:44:28 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/11/20 07:44:29 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


csv               734.2 KB
csv.gz            138.2 KB
json             1433.4 KB
json.gz           156.0 KB
avro (none)       383.2 KB
avro (snappy)     235.6 KB


**Why Kafka and Iceberg manifests use Avro:** each record is written whole, one after another,
so appending a row is cheap, and the schema travels with the data.

**Lesson → Stage 3:** row-based storage means an analytics query reads every column, even when it needs only 2.

# Stage 3: Columnar files (ORC vs Parquet)
*Fixes: reading whole rows. Data is stored per column, with min/max statistics.*

Uses `pyarrow` (already in `requirements.txt`) and the `download_first` helper from Stage 2.

In [11]:
PARQUET = f"{BASE}/files/parquet"
ORC     = f"{BASE}/files/orc"

import pyarrow.parquet as pq
import pyarrow.orc as po

In [7]:
# 3.1 Write the same data as Parquet and as ORC
df.coalesce(1).write.mode("overwrite").parquet(f"{PARQUET}/data")
df.coalesce(1).write.mode("overwrite").orc(f"{ORC}/data")

print(f"parquet {tree(f'{PARQUET}/data')/1024:8.1f} KB")
print(f"orc     {tree(f'{ORC}/data')/1024:8.1f} KB")

print("PARQUET read back"); spark.read.parquet(f"{PARQUET}/data").printSchema()
print("ORC read back");     spark.read.orc(f"{ORC}/data").printSchema()

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


parquet    219.0 KB
orc         26.0 KB
PARQUET read back


root
 |-- id: integer (nullable = true)
 |-- big_id: long (nullable = true)
 |-- customer: string (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- sale_ts: timestamp (nullable = true)
 |-- is_paid: boolean (nullable = true)

ORC read back
root
 |-- id: integer (nullable = true)
 |-- big_id: long (nullable = true)
 |-- customer: string (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- sale_ts: timestamp (nullable = true)
 |-- is_paid: boolean (nullable = true)



### 3.2 Parquet file layout (diagram: `~/Desktop/Screenshot 2026-11-16 at 3.11.19 PM.png`, from parquet.apache.org)
```
File                                         Footer
├─ Magic Number (4 bytes) "PAR1"             ├─ FileMetaData (Thrift compact protocol)
├─ Row group 0                               │   ├─ version, schema, extra key/value pairs
│   ├─ Column a                              │   ├─ Row group 0 meta data
│   │   ├─ Page 0                            │   │   ├─ Column a meta data
│   │   │   ├─ Page header (Thrift)          │   │   │   type / path / encodings / codec
│   │   │   ├─ Repetition levels             │   │   │   num values
│   │   │   ├─ Definition levels             │   │   │   offset of first data page   ──> Page 0
│   │   │   └─ values                        │   │   │   offset of first index page
│   │   └─ Page 1 ...                        │   │   │   compressed / uncompressed size
│   └─ Column b ...                          │   │   │   statistics (min / max / null count)
├─ Row group 1 ...                           │   │   └─ Column b meta data ...
│                                            │   └─ Row group 1 meta data ...
│  (newer files) Page Index                  ├─ Footer length (4 bytes)
│   ColumnIndex + OffsetIndex per chunk      └─ Magic Number (4 bytes) "PAR1"
```
| Diagram box | What it holds | Cell |
|---|---|---|
| Magic number, footer length | `PAR1` at both ends; 4-byte length of the footer | 3.2b |
| FileMetaData | format version, schema, total rows, writer, key/value pairs (Spark's schema) | 3.2c |
| Row group / column meta data | type, path, encodings, codec, num values, page offsets, sizes, **min/max/null count** | 3.2d |
| Page header | page type, sizes, num values, encoding, crc (old writers: page min/max too) | 3.2e |
| Repetition / definition levels | nested position / which values are null. Flat columns have no repetition levels; `NOT NULL` columns have no definition levels | 3.8 |
| Page Index *(not in the diagram, added later)* | min/max/null count **per page** + where each page starts, stored near the footer | 3.2e |

**Why the footer is at the end:** the writer streams the data in one pass and writes the metadata last.
A reader does the opposite: read the last 8 bytes → footer length → footer → jump straight to the chunks/pages it needs.

In [12]:
# 3.2a Write a file with several row groups, and several pages per column chunk
# parquet.block.size = row group size, parquet.page.size = page size (both in bytes)
(df.coalesce(1).write.mode("overwrite")
   .option("parquet.block.size", 64 * 1024)      # ~64 KB row groups
   .option("parquet.page.size",  8 * 1024)       # ~8 KB pages
   .parquet(f"{PARQUET}/small_rowgroups"))

local = download_first(f"{PARQUET}/small_rowgroups", ".parquet", "/tmp/sample.parquet")

In [13]:
# 3.2b Start and end of the file: magic number + footer length
import struct

with open(local, "rb") as f:
    data = f.read()

footer_len = struct.unpack("<I", data[-8:-4])[0]     # 4 bytes, little endian
print("file size:        ", len(data), "bytes")
print("first 4 bytes:    ", data[:4])                # b'PAR1'
print("last 4 bytes:     ", data[-4:])               # b'PAR1'
print("footer length:    ", footer_len, "bytes")
print("footer starts at: ", len(data) - 8 - footer_len)

file size:         248449 bytes
first 4 bytes:     b'PAR1'
last 4 bytes:      b'PAR1'
footer length:     3417 bytes
footer starts at:  245024


In [14]:
# 3.2c File level (footer): schema, rows, row groups, writer, key/value metadata
pf = pq.ParquetFile(local)
meta = pf.metadata

print(meta)                 # num_rows, num_row_groups, created_by
print(pf.schema)            # physical types (INT32, BYTE_ARRAY...) + logical types (DECIMAL, DATE...)

for k, v in meta.metadata.items():      # Spark saves its own schema here
    print(k.decode(), "=>", v.decode()[:200], "...")

  created_by: parquet-mr version 1.16.0 (build 402c3810c372d29603e181771acebfecc71bef61)
  num_columns: 7
  num_rows: 10000
  num_row_groups: 4
  format_version: 1.0
  serialized_size: 3417
required group field_id=-1 spark_schema {
  required int32 field_id=-1 id;
  required int64 field_id=-1 big_id;
  optional binary field_id=-1 customer (String);
  optional int64 field_id=-1 amount (Decimal(precision=10, scale=2));
  optional int32 field_id=-1 sale_date (Date);
  required int96 field_id=-1 sale_ts;
  optional boolean field_id=-1 is_paid;
}

org.apache.spark.sql.parquet.row.metadata => {"type":"struct","fields":[{"name":"id","type":"integer","nullable":false,"metadata":{}},{"name":"big_id","type":"long","nullable":false,"metadata":{}},{"name":"customer","type":"string","nullable":tr ...
org.apache.spark.version => 4.1.0 ...


In [15]:
# 3.2d Row group / column chunk level (footer): one entry per column per row group
for rg in range(meta.num_row_groups):
    g = meta.row_group(rg)
    print(f"ROW GROUP {rg}: rows={g.num_rows}  bytes={g.total_byte_size}")
    for c in range(g.num_columns):
        col = g.column(c)
        s = col.statistics
        print(f"   {col.path_in_schema:10} {col.physical_type:10} codec={col.compression:7} "
              f"size={col.total_compressed_size:6}  "
              f"min={s.min if s and s.has_min_max else None}  max={s.max if s and s.has_min_max else None}  "
              f"nulls={s.null_count if s else None}")
    print()

ROW GROUP 0: rows=2858  bytes=106308
   id         INT32      codec=SNAPPY  size= 11493  min=0  max=2857  nulls=0
   big_id     INT64      codec=SNAPPY  size= 17378  min=0  max=2857000000  nulls=0
   customer   BYTE_ARRAY codec=SNAPPY  size=  5489  min=customer_1  max=customer_99  nulls=286
   amount     INT64      codec=SNAPPY  size= 12030  min=1.36  max=999.98  nulls=286
   sale_date  INT32      codec=SNAPPY  size=   218  min=2026-01-01  max=2026-01-05  nulls=0
   sale_ts    INT96      codec=SNAPPY  size= 22074  min=None  max=None  nulls=0
   is_paid    BOOLEAN    codec=SNAPPY  size=    55  min=False  max=True  nulls=0

ROW GROUP 1: rows=2858  bytes=106285
   id         INT32      codec=SNAPPY  size= 11494  min=2858  max=5715  nulls=0
   big_id     INT64      codec=SNAPPY  size= 17386  min=2858000000  max=5715000000  nulls=0
   customer   BYTE_ARRAY codec=SNAPPY  size=  5493  min=customer_1  max=customer_99  nulls=286
   amount     INT64      codec=SNAPPY  size= 11988  min=0.46  max=

In [17]:
# 3.2d (cont.) Where a column chunk sits in the file and how it is encoded
col = meta.row_group(0).column(0)       # row group 0, column `id`
print("path:                  ", col.path_in_schema)
print("encodings:             ", col.encodings)
print("has dictionary page:   ", col.has_dictionary_page, col.dictionary_page_offset)
print("first data page offset:", col.data_page_offset)             # arrow in the diagram -> Page 0
# print("first index page:      ", col.has_index_page, col.index_page_offset)   # diagram box; unused in practice (None)
print("compressed size:       ", col.total_compressed_size)
print("uncompressed size:     ", col.total_uncompressed_size)
print("values:                ", col.num_values)

path:                   id
encodings:              ('BIT_PACKED', 'PLAIN')
has dictionary page:    False None
first data page offset: 4
compressed size:        11493
uncompressed size:      11484
values:                 2858


In [18]:
# 3.2e Page level: the Page Index (ColumnIndex + OffsetIndex)
# pyarrow doesn't show it, so we use Parquet's Java reader (parquet-mr), already in Spark's jars.
jvm  = spark._jvm
conf = spark._jsc.hadoopConfiguration()

folder = jvm.org.apache.hadoop.fs.Path(f"{PARQUET}/small_rowgroups")
file   = [st.getPath() for st in folder.getFileSystem(conf).listStatus(folder)
          if st.getPath().getName().endswith(".parquet")][0]

reader = jvm.org.apache.parquet.hadoop.ParquetFileReader.open(
    jvm.org.apache.parquet.hadoop.util.HadoopInputFile.fromPath(file, conf))

chunk = reader.getFooter().getBlocks().get(0).getColumns().get(0)    # row group 0, column `id`
print("column:", chunk.getPath())

print("\nOffsetIndex: where each page starts and its first row")
print(reader.readOffsetIndex(chunk))

print("ColumnIndex: min / max / null count of each page")
print(reader.readColumnIndex(chunk))

reader.close()

column: [id]

OffsetIndex: where each page starts and its first row
                          offset       compressed size       first row index       unencoded bytes
page-0                         4                  7674                     0                     -
page-1                      7678                  3819                  1911                     -

ColumnIndex: min / max / null count of each page
Boundary order: ASCENDING
                      null count  min                                       max                                        rep level histogram   def level histogram
page-0                         0  0                                         1910                                                    <none>                <none>
page-1                         0  1911                                      2857                                                    <none>                <none>



**What to notice**
- `PAR1` at both ends; the footer length sits right before the last `PAR1`
- Footer stats are per **column chunk** (one per row group); the Page Index has stats per **page**
- Filter `id < 100`: the footer lets Spark skip whole row groups, the Page Index lets it skip pages inside a row group
- Spark's schema is saved in the footer's key/value metadata (`org.apache.spark.sql.parquet.row.metadata`)

In [19]:
# 3.3 ORC internals: file -> stripes (index + data + stripe footer) -> file footer -> postscript
# Small stripes and a small row index stride so there is more than one of each.
(df.coalesce(1).write.mode("overwrite")
   .option("orc.stripe.size", 64 * 1024)
   .option("orc.row.index.stride", 1000)          # min/max index entry every 1000 rows
   .orc(f"{ORC}/small_stripes"))

local = download_first(f"{ORC}/small_stripes", ".orc", "/tmp/sample.orc")
orc = po.ORCFile(local)

print("rows:            ", orc.nrows)
print("stripes:         ", orc.nstripes)
print("compression:     ", orc.compression)
print("row index stride:", orc.row_index_stride)
print("writer:          ", orc.writer, orc.software_version)
print("footer bytes:    ", orc.file_footer_length)
print("schema:\n", orc.schema)

# each stripe can be read on its own
print("stripe 0 rows:", orc.read_stripe(0).num_rows)

26/11/20 10:44:27 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


rows:             10000
stripes:          2
compression:      ZSTD
row index stride: 1000
writer:           ORC_JAVA ORC Java 2.2.1
footer bytes:     444
schema:
 id: int32
big_id: int64
customer: string
  -- field metadata --
  spark.sql.catalyst.type: 'string'
amount: decimal128(10, 2)
sale_date: date32[day]
sale_ts: timestamp[ns]
  -- field metadata --
  spark.sql.catalyst.type: 'timestamp'
is_paid: bool
-- schema metadata --
org.apache.spark.version: '4.1.0'
stripe 0 rows: 5120


26/11/20 10:58:19 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 171264 ms exceeds timeout 120000 ms
26/11/20 10:58:21 ERROR Inbox: Ignoring error
java.lang.AssertionError: assertion failed: BlockManager re-registration shouldn't succeed when the executor is lost
	at scala.Predef$.assert(Predef.scala:279)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$register(BlockManagerMasterEndpoint.scala:748)
	at org.apache.spark.storage.BlockManagerMasterEndpoint$$anonfun$receiveAndReply$1.applyOrElse(BlockManagerMasterEndpoint.scala:141)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:104)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:216)
	at org.apache.spark.rpc.netty.Inbox.process(Inbox.scala:101)
	at org.apache.spark.rpc.netty.MessageLoop.org$apache$spark$rpc$netty$MessageLoop$$receiveLoop(MessageLoop.scala:76)
	at org.apache.spark.rpc.netty.MessageLoop$$anon$1.run(MessageL

**What to notice**
- Parquet: *row group* → one *column chunk* per column → pages. Stats per column chunk in the footer
- ORC: *stripe* → index (min/max every 1000 rows) + column data. Stats per file, per stripe, per row group
- Both keep the schema and types in the file (like Avro), but group values by column

In [20]:
# 3.4 Codecs: none vs snappy vs zstd
rows = []
for fmt in ["parquet", "orc"]:
    for codec in ["none", "snappy", "zstd"]:
        path = f"{BASE}/files/{fmt}/codec_{codec}"
        df.coalesce(1).write.mode("overwrite").format(fmt).option("compression", codec).save(path)
        rows.append((fmt, codec, round(tree(path) / 1024, 1)))

spark.createDataFrame(rows, "format string, codec string, size_kb double").show()

26/11/20 11:07:21 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/11/20 11:07:24 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/11/20 11:07:25 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


+-------+------+-------+
| format| codec|size_kb|
+-------+------+-------+
|parquet|  none|  349.0|
|parquet|snappy|  219.0|
|parquet|  zstd|  108.3|
|    orc|  none|   75.0|
|    orc|snappy|   32.8|
|    orc|  zstd|   26.0|
+-------+------+-------+



In [21]:
# 3.5 Dictionary encoding: low vs high cardinality
# low_card has 5 distinct values: stored once in a dictionary, then small integer codes per row.
# high_card is different on every row: a dictionary doesn't help.
card = df.select(
    F.concat(F.lit("customer_"), F.col("id") % 5).alias("low_card"),
    F.expr("uuid()").alias("high_card"),
)

sizes = {}
for column in ["low_card", "high_card"]:
    for dictionary in [True, False]:
        path = f"{PARQUET}/dict_{column}_{dictionary}"
        (card.select(column).coalesce(1).write.mode("overwrite")
             .option("parquet.enable.dictionary", dictionary)
             .option("compression", "none")                  # so only the encoding makes the difference
             .parquet(path))
        sizes[f"{column:9} dictionary={dictionary}"] = tree(path)

for name, size in sizes.items():
    print(f"{name:30} {size/1024:8.1f} KB")

local = download_first(f"{PARQUET}/dict_low_card_True", ".parquet", "/tmp/low.parquet")
print("encodings:", pq.ParquetFile(local).metadata.row_group(0).column(0).encodings)   # RLE_DICTIONARY

low_card  dictionary=True           4.3 KB
low_card  dictionary=False        137.2 KB
high_card dictionary=True         391.2 KB
high_card dictionary=False        391.2 KB
encodings: ('BIT_PACKED', 'PLAIN_DICTIONARY', 'RLE')


In [22]:
# 3.6 Column pruning: read only 2 columns
# ReadSchema in the plan lists only the columns the query needs. The other column chunks are never read.
spark.read.parquet(f"{PARQUET}/data").select("id", "amount").explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [id#61,amount#64] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[s3a://format-lab/files/parquet/data], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<id:int,amount:decimal(10,2)>




In [23]:
# 3.7 Predicate pushdown: filter on id
# PushedFilters in the plan = the filter is given to the file reader, which compares it with min/max
# stats and skips row groups / stripes that can't match.
# Spark UI (http://localhost:4040 -> SQL tab): compare "number of output rows" of the Scan with the filter on and off.
q_parquet = spark.read.parquet(f"{PARQUET}/small_rowgroups").filter("id < 100").select("id", "amount")
q_parquet.explain()          # PushedFilters: [IsNotNull(id), LessThan(id,100)]
print("parquet rows:", q_parquet.count())

q_orc = spark.read.orc(f"{ORC}/small_stripes").filter("id < 100").select("id", "amount")
q_orc.explain()
print("orc rows:", q_orc.count())

== Physical Plan ==
*(1) Filter (isnotnull(id#69) AND (id#69 < 100))
+- *(1) ColumnarToRow
   +- FileScan parquet [id#69,amount#72] Batched: true, DataFilters: [isnotnull(id#69), (id#69 < 100)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[s3a://format-lab/files/parquet/small_rowgroups], PartitionFilters: [], PushedFilters: [IsNotNull(id), LessThan(id,100)], ReadSchema: struct<id:int,amount:decimal(10,2)>




parquet rows: 100
== Physical Plan ==
*(1) Filter (isnotnull(id#83) AND (id#83 < 100))
+- *(1) ColumnarToRow
   +- FileScan orc [id#83,amount#86] Batched: true, DataFilters: [isnotnull(id#83), (id#83 < 100)], Format: ORC, Location: InMemoryFileIndex(1 paths)[s3a://format-lab/files/orc/small_stripes], PartitionFilters: [], PushedFilters: [IsNotNull(id), LessThan(id,100)], ReadSchema: struct<id:int,amount:decimal(10,2)>




orc rows: 100


In [24]:
# 3.7 (cont.) Same query with pushdown turned off: the Scan now reads all 10,000 rows
spark.conf.set("spark.sql.parquet.filterPushdown", False)
spark.read.parquet(f"{PARQUET}/small_rowgroups").filter("id < 100").count()
spark.conf.set("spark.sql.parquet.filterPushdown", True)

In [25]:
# 3.8 Nested types (struct / array) in both formats
df_nested = (df
    .withColumn("address", F.struct(F.lit("Lahore").alias("city"), (F.col("id") % 100).alias("zip")))
    .withColumn("tags", F.array(F.lit("a"), F.lit("b"))))

df_nested.coalesce(1).write.mode("overwrite").parquet(f"{PARQUET}/nested")
df_nested.coalesce(1).write.mode("overwrite").orc(f"{ORC}/nested")

spark.read.parquet(f"{PARQUET}/nested").select("id", "address.city", "tags").show(3)
spark.read.orc(f"{ORC}/nested").printSchema()

# Parquet stores every nested leaf (address.city, address.zip, tags.list.element) as its own column
local = download_first(f"{PARQUET}/nested", ".parquet", "/tmp/nested.parquet")
print(pq.ParquetFile(local).schema)

# Repetition / definition levels (the two boxes under the page header in the diagram)
#   max_definition_level > 0  -> column can be null (or sits inside something nullable)
#   max_repetition_level > 0  -> column is inside a list (array), values can repeat per row
schema = pq.ParquetFile(local).schema
for i in range(len(schema)):
    col = schema.column(i)
    print(f"{col.path:25} definition={col.max_definition_level}  repetition={col.max_repetition_level}")


26/11/20 11:13:34 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


+---+------+------+
| id|  city|  tags|
+---+------+------+
|  0|Lahore|[a, b]|
|  1|Lahore|[a, b]|
|  2|Lahore|[a, b]|
+---+------+------+
only showing top 3 rows
root
 |-- id: integer (nullable = true)
 |-- big_id: long (nullable = true)
 |-- customer: string (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- sale_ts: timestamp (nullable = true)
 |-- is_paid: boolean (nullable = true)
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- zip: integer (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)

required group field_id=-1 spark_schema {
  required int32 field_id=-1 id;
  required int64 field_id=-1 big_id;
  optional binary field_id=-1 customer (String);
  optional int64 field_id=-1 amount (Decimal(precision=10, scale=2));
  optional int32 field_id=-1 sale_date (Date);
  required int96 field_id=-1 sale_ts;
  optional boolean field_id=-1 is

**What to notice**
- Columnar + compression = much smaller than CSV / JSON / Avro (compare with 1.4 and 2.5)
- Reading 2 columns reads only those column chunks; a filter can skip whole row groups / stripes
- ORC is common in Hive, Parquet in Spark / Delta / Iceberg; the ideas are the same

**Lesson → Stage 4:** a single file isn't a dataset. Large data needs many files and a layout.

# Stage 4: Directory layout with raw Spark (no catalog)
*Fixes: one big file. Data is split into folders and files.*

In [5]:
LAYOUT = f"{BASE}/files/layout"

# helper: list the files under a path (relative name + size), so we can count and look at them
def ls_files(path):
    jvm = spark._jvm
    p = jvm.org.apache.hadoop.fs.Path(path)
    fs = p.getFileSystem(spark._jsc.hadoopConfiguration())
    if not fs.exists(p):
        return []
    it, out = fs.listFiles(p, True), []
    while it.hasNext():
        f = it.next()
        out.append((f.getPath().toString().replace(path + "/", ""), f.getLen()))
    return out

def show_files(path, limit=20):
    files = ls_files(path)
    for name, size in files[:limit]:
        print(f"{size/1024:8.1f} KB  {name}")
    if len(files) > limit:
        print(f"... {len(files) - limit} more")
    print(f"{len(files)} files")

In [27]:
# 4.1 partitionBy("sale_date"): one folder per value, Hive-style name `sale_date=2026-01-01/`
df.write.mode("overwrite").partitionBy("sale_date").parquet(f"{LAYOUT}/by_date")
show_files(f"{LAYOUT}/by_date")

     0.0 KB  _SUCCESS
    21.1 KB  sale_date=2026-01-01/part-00000-3db1c6a7-52a7-4db1-8e44-eb797dec9224.c000.snappy.parquet
    21.1 KB  sale_date=2026-01-01/part-00001-3db1c6a7-52a7-4db1-8e44-eb797dec9224.c000.snappy.parquet
    23.7 KB  sale_date=2026-01-02/part-00000-3db1c6a7-52a7-4db1-8e44-eb797dec9224.c000.snappy.parquet
    23.7 KB  sale_date=2026-01-02/part-00001-3db1c6a7-52a7-4db1-8e44-eb797dec9224.c000.snappy.parquet
    23.7 KB  sale_date=2026-01-03/part-00000-3db1c6a7-52a7-4db1-8e44-eb797dec9224.c000.snappy.parquet
    23.7 KB  sale_date=2026-01-03/part-00001-3db1c6a7-52a7-4db1-8e44-eb797dec9224.c000.snappy.parquet
    23.6 KB  sale_date=2026-01-04/part-00000-3db1c6a7-52a7-4db1-8e44-eb797dec9224.c000.snappy.parquet
    23.7 KB  sale_date=2026-01-04/part-00001-3db1c6a7-52a7-4db1-8e44-eb797dec9224.c000.snappy.parquet
    23.7 KB  sale_date=2026-01-05/part-00000-3db1c6a7-52a7-4db1-8e44-eb797dec9224.c000.snappy.parquet
    23.7 KB  sale_date=2026-01-05/part-00001-3db1c6a7-52a7-4

In [28]:
# 4.1 (cont.) The partition column is NOT inside the files. It comes back from the folder name.
local = download_first(f"{LAYOUT}/by_date/sale_date=2026-01-01", ".parquet", "/tmp/part.parquet")
print("columns in the file:", pq.ParquetFile(local).schema_arrow.names)

spark.read.parquet(f"{LAYOUT}/by_date").printSchema()      # sale_date is added back, from the folder

columns in the file: ['id', 'big_id', 'customer', 'amount', 'sale_ts', 'is_paid']
root
 |-- id: integer (nullable = true)
 |-- big_id: long (nullable = true)
 |-- customer: string (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- sale_ts: timestamp (nullable = true)
 |-- is_paid: boolean (nullable = true)
 |-- sale_date: date (nullable = true)



In [29]:
# 4.2 Partition pruning: filter on the partition column -> only one folder is read
# PartitionFilters in the plan; inputFiles() lists only files from sale_date=2026-01-02.
q = spark.read.parquet(f"{LAYOUT}/by_date").filter("sale_date = '2026-01-02'")
q.explain()
for f in q.inputFiles():
    print(f.replace(f"{LAYOUT}/by_date/", ""))

# filter on a normal column: every folder is read
print("filter on id -> files read:", len(spark.read.parquet(f"{LAYOUT}/by_date").filter("id = 5").inputFiles()))

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [id#166,big_id#167L,customer#168,amount#169,sale_ts#170,is_paid#171,sale_date#172] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[s3a://format-lab/files/layout/by_date], PartitionFilters: [isnotnull(sale_date#172), (sale_date#172 = 2026-01-02)], PushedFilters: [], ReadSchema: struct<id:int,big_id:bigint,customer:string,amount:decimal(10,2),sale_ts:timestamp,is_paid:boolean>


sale_date=2026-01-01/part-00001-3db1c6a7-52a7-4db1-8e44-eb797dec9224.c000.snappy.parquet
sale_date=2026-01-05/part-00000-3db1c6a7-52a7-4db1-8e44-eb797dec9224.c000.snappy.parquet
sale_date=2026-01-02/part-00001-3db1c6a7-52a7-4db1-8e44-eb797dec9224.c000.snappy.parquet
sale_date=2026-01-03/part-00001-3db1c6a7-52a7-4db1-8e44-eb797dec9224.c000.snappy.parquet
sale_date=2026-01-05/part-00001-3db1c6a7-52a7-4db1-8e44-eb797dec9224.c000.snappy.parquet
sale_date=2026-01-01/part-00000-3db1c6a7-52a7-4db1-8e44-eb797dec9224.c000.snap

In [30]:
# 4.3 Small-files problem
# Every task writes its own file. 200 tasks = 200 files; with partitionBy, up to 200 files per folder.
df.repartition(200).write.mode("overwrite").parquet(f"{LAYOUT}/many_files")
df.coalesce(4).write.mode("overwrite").parquet(f"{LAYOUT}/few_files")
df.repartition(200).write.mode("overwrite").partitionBy("sale_date").parquet(f"{LAYOUT}/many_files_partitioned")

for name in ["many_files", "few_files", "many_files_partitioned"]:
    files = [f for f in ls_files(f"{LAYOUT}/{name}") if f[0].endswith(".parquet")]
    avg_kb = sum(s for _, s in files) / len(files) / 1024
    print(f"{name:24} {len(files):5} files   avg {avg_kb:6.1f} KB per file")

many_files                 200 files   avg    3.4 KB per file
few_files                    2 files   avg  112.0 KB per file
many_files_partitioned    1000 files   avg    1.9 KB per file


In [31]:
# 4.3 (cont.) Common fix: repartition by the partition column -> one file per folder
df.repartition("sale_date").write.mode("overwrite").partitionBy("sale_date").parquet(f"{LAYOUT}/one_file_per_folder")
show_files(f"{LAYOUT}/one_file_per_folder")

     0.0 KB  _SUCCESS
    39.7 KB  sale_date=2026-01-01/part-00000-214bb344-09a9-47d5-9c9a-7d45ea2f0849.c000.snappy.parquet
    44.5 KB  sale_date=2026-01-02/part-00000-214bb344-09a9-47d5-9c9a-7d45ea2f0849.c000.snappy.parquet
    44.5 KB  sale_date=2026-01-03/part-00000-214bb344-09a9-47d5-9c9a-7d45ea2f0849.c000.snappy.parquet
    44.4 KB  sale_date=2026-01-04/part-00000-214bb344-09a9-47d5-9c9a-7d45ea2f0849.c000.snappy.parquet
    44.5 KB  sale_date=2026-01-05/part-00000-214bb344-09a9-47d5-9c9a-7d45ea2f0849.c000.snappy.parquet
6 files


**What to notice**
- Each file costs a list call + open + footer read. Thousands of tiny files are slower than a few big ones
- Too many partition values (e.g. partition by `id`) = too many folders and tiny files

In [ ]:
# 4.4 The _SUCCESS marker
# Written last, when the job finishes. Spark readers don't check it: delete it and the data still reads.
show_files(f"{LAYOUT}/few_files")

jvm = spark._jvm
p = jvm.org.apache.hadoop.fs.Path(f"{LAYOUT}/few_files/_SUCCESS")
p.getFileSystem(spark._jsc.hadoopConfiguration()).delete(p, False)

print("after deleting _SUCCESS, rows:", spark.read.parquet(f"{LAYOUT}/few_files").count())

In [ ]:
# 4.5 A failed overwrite
# Spark deletes the old files first, then writes the new ones. Make the job fail halfway:
# the old data is already gone, and nothing tells a reader the folder is broken.
path = f"{LAYOUT}/failed_write"
df.coalesce(1).write.mode("overwrite").parquet(path)
print("before:", spark.read.parquet(path).count(), "rows")

bad = df.withColumn("id", F.when(F.col("id") == 9_000, F.raise_error("boom")).otherwise(F.col("id")))
try:
    bad.repartition(4).write.mode("overwrite").parquet(path)
except Exception as e:
    print("write failed:", str(e).splitlines()[0][:120])

show_files(path)       # old files gone; maybe a leftover _temporary/ folder, no _SUCCESS
try:
    print("after:", spark.read.parquet(path).count(), "rows")
except Exception as e:
    print("read failed:", str(e).splitlines()[0][:120])

In [ ]:
# 4.6 overwrite vs append: no history
# append adds new files next to the old ones; overwrite deletes everything first. There is no undo.
path = f"{LAYOUT}/modes"

df.limit(10).coalesce(1).write.mode("overwrite").parquet(path)
print("first write:", spark.read.parquet(path).count(), "rows"); show_files(path)

df.limit(10).coalesce(1).write.mode("append").parquet(path)
print("\nafter append:", spark.read.parquet(path).count(), "rows"); show_files(path)

df.limit(3).coalesce(1).write.mode("overwrite").parquet(path)
print("\nafter overwrite:", spark.read.parquet(path).count(), "rows"); show_files(path)

In [ ]:
# 4.7 Nothing records path, format or schema
# To read the data you must already know all three. Wrong format = error; the folder says nothing about it.
try:
    spark.read.orc(f"{LAYOUT}/by_date").count()
except Exception as e:
    print("read as ORC failed:", str(e).splitlines()[0][:120])

**What to notice**
- The "dataset" is just a folder of files plus naming rules (`sale_date=...`)
- No atomic commit: a failed or running write is visible to readers
- No history, no record of the schema/format: every reader must know them

**Lesson → Stage 5:** to read the data you must know its path, format and schema. Nothing records them.

# Stage 5: Hive tables via Spark (lakehouse HMS)
*Fixes: knowing path/format/schema. The **metastore** records them; the table is still a directory.*

- Tables go in `spark_catalog` (Spark's Hive catalog), database `fmt_spark`, files under `format-lab/spark-hive/`
- The default catalog is still `lakehouse` (Iceberg), so table names are written in full: `spark_catalog.fmt_spark.<table>`
- Needs: `.enableHiveSupport()` in the session cell and `psycopg2-binary` (FORMAT_LAB block in `requirements.txt`)

In [6]:
SPARK_HIVE = f"{BASE}/spark-hive"
DB = "spark_catalog.fmt_spark"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB} LOCATION '{SPARK_HIVE}/fmt_spark.db'")

df.createOrReplaceTempView("sales_src")       # source data for every table below
COLS = "id, big_id, customer, amount, sale_ts, is_paid"   # all columns except the partition column

In [7]:
# helper: run SQL on the metastore database (hive-postgres) and get a pandas DataFrame
# HMS tables and columns are UPPERCASE, so they need double quotes in Postgres: "TBLS", "TBL_NAME"
import psycopg2, pandas as pd

def hms(sql):
    with psycopg2.connect(host="hive-postgres", port=5432, dbname="metastore",
                          user="hive", password="hive") as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

# one query we reuse: every table of a database, with its type, location, format and serde
def hms_tables(db):
    return hms(f"""
        SELECT t."TBL_NAME", t."TBL_TYPE", s."LOCATION", s."INPUT_FORMAT", se."SLIB" AS serde
        FROM "TBLS" t
        JOIN "DBS" d     ON t."DB_ID" = d."DB_ID"
        JOIN "SDS" s     ON t."SD_ID" = s."SD_ID"
        JOIN "SERDES" se ON s."SERDE_ID" = se."SERDE_ID"
        WHERE d."NAME" = '{db}'
        ORDER BY t."TBL_NAME"
    """)

hms('SELECT "DB_ID", "NAME", "DB_LOCATION_URI" FROM "DBS"')

,DB_ID,NAME,DB_LOCATION_URI
0,1,default,s3a://buck1/warehouse
1,11,demo,s3://buck1/warehouse/demo.db
2,31,fmt_spark,s3a://format-lab/spark-hive/fmt_spark.db


In [8]:
# 5.1 Managed vs external table
# managed  = the metastore owns the files: DROP TABLE deletes them
# external = the metastore only points at the files: DROP TABLE keeps them
spark.sql(f"DROP TABLE IF EXISTS {DB}.sales_managed")
spark.sql(f"DROP TABLE IF EXISTS {DB}.sales_external")

spark.sql(f"CREATE TABLE {DB}.sales_managed STORED AS PARQUET AS SELECT * FROM sales_src")
spark.sql(f"""
    CREATE EXTERNAL TABLE {DB}.sales_external
    STORED AS PARQUET LOCATION '{SPARK_HIVE}/external/sales_external'
    AS SELECT * FROM sales_src
""")

spark.sql(f"DESCRIBE TABLE EXTENDED {DB}.sales_managed").filter("col_name in ('Type', 'Location', 'Provider')").show(truncate=False)
spark.sql(f"DESCRIBE TABLE EXTENDED {DB}.sales_external").filter("col_name in ('Type', 'Location', 'Provider')").show(truncate=False)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/11/23 05:04:41 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


+--------+------------------------------------------------------+-------+
|col_name|data_type                                             |comment|
+--------+------------------------------------------------------+-------+
|Type    |EXTERNAL                                              |       |
|Provider|hive                                                  |       |
|Location|s3a://format-lab/spark-hive/fmt_spark.db/sales_managed|       |
+--------+------------------------------------------------------+-------+

+--------+---------------------------------------------------+-------+
|col_name|data_type                                          |comment|
+--------+---------------------------------------------------+-------+
|Type    |EXTERNAL                                           |       |
|Provider|hive                                               |       |
|Location|s3a://format-lab/spark-hive/external/sales_external|       |
+--------+---------------------------------------------

In [9]:
# 5.1 (cont.) What the metastore stored. Check TBL_TYPE of the managed table:
# Hive 4 may save a non-ACID managed table as EXTERNAL_TABLE + external.table.purge=true
display(hms_tables("fmt_spark"))

hms("""
    SELECT t."TBL_NAME", p."PARAM_KEY", p."PARAM_VALUE"
    FROM "TABLE_PARAMS" p JOIN "TBLS" t ON p."TBL_ID" = t."TBL_ID"
    WHERE t."TBL_NAME" IN ('sales_managed', 'sales_external')
      AND p."PARAM_KEY" IN ('EXTERNAL', 'external.table.purge', 'TRANSLATED_TO_EXTERNAL')
""")

,TBL_NAME,TBL_TYPE,LOCATION,INPUT_FORMAT,serde
0,sales_external,EXTERNAL_TABLE,s3a://format-lab/spark-hive/external/sales_ext...,org.apache.hadoop.hive.ql.io.parquet.MapredPar...,org.apache.hadoop.hive.ql.io.parquet.serde.Par...
1,sales_managed,EXTERNAL_TABLE,s3a://format-lab/spark-hive/fmt_spark.db/sales...,org.apache.hadoop.hive.ql.io.parquet.MapredPar...,org.apache.hadoop.hive.ql.io.parquet.serde.Par...


,TBL_NAME,PARAM_KEY,PARAM_VALUE
0,sales_managed,EXTERNAL,TRUE
1,sales_managed,external.table.purge,TRUE
2,sales_managed,TRANSLATED_TO_EXTERNAL,TRUE
3,sales_external,EXTERNAL,TRUE


In [10]:
# 5.1 (cont.) DROP both tables, then look at the files
spark.sql(f"DROP TABLE {DB}.sales_managed")
spark.sql(f"DROP TABLE {DB}.sales_external")

print("managed folder:");  show_files(f"{SPARK_HIVE}/fmt_spark.db/sales_managed")   # gone
print("external folder:"); show_files(f"{SPARK_HIVE}/external/sales_external")     # still there

managed folder:
0 files
external folder:
     0.0 KB  _SUCCESS
   112.1 KB  part-00000-9e591e50-6aec-40ed-abf0-fb42f9aa1bb3-c000.snappy.parquet
   111.9 KB  part-00001-9e591e50-6aec-40ed-abf0-fb42f9aa1bb3-c000.snappy.parquet
3 files


In [11]:
# 5.2 STORED AS TEXTFILE / ORC / PARQUET / AVRO: same data, 4 tables
for fmt in ["textfile", "orc", "parquet", "avro"]:
    spark.sql(f"DROP TABLE IF EXISTS {DB}.sales_{fmt}")
    row_format = "ROW FORMAT DELIMITED FIELDS TERMINATED BY ','" if fmt == "textfile" else ""
    spark.sql(f"CREATE TABLE {DB}.sales_{fmt} {row_format} STORED AS {fmt} AS SELECT * FROM sales_src")

for fmt in ["textfile", "orc", "parquet", "avro"]:
    print(f"--- sales_{fmt}"); show_files(f"{SPARK_HIVE}/fmt_spark.db/sales_{fmt}", limit=3)

hms_tables("fmt_spark")      # different INPUT_FORMAT and SERDE per table

26/11/23 05:07:51 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/11/23 05:07:53 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/11/23 05:07:56 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


--- sales_textfile
   343.5 KB  part-00000-fa85953a-059a-4d2d-a3b2-dcd3ae70eced-c000
   345.7 KB  part-00001-fa85953a-059a-4d2d-a3b2-dcd3ae70eced-c000
2 files
--- sales_orc
     0.0 KB  _SUCCESS
    14.3 KB  part-00000-27410f31-610c-4e08-8819-2965d59a8ad8-c000.zstd.orc
    14.3 KB  part-00001-27410f31-610c-4e08-8819-2965d59a8ad8-c000.zstd.orc
3 files
--- sales_parquet
     0.0 KB  _SUCCESS
   112.1 KB  part-00000-3b81a0d6-8647-468c-bd30-00c4d8dba99b-c000.snappy.parquet
   111.9 KB  part-00001-3b81a0d6-8647-468c-bd30-00c4d8dba99b-c000.snappy.parquet
3 files
--- sales_avro
   190.0 KB  part-00000-fea9b784-cec5-43e1-a8fe-57fd3f09b256-c000
   191.9 KB  part-00001-fea9b784-cec5-43e1-a8fe-57fd3f09b256-c000
2 files


,TBL_NAME,TBL_TYPE,LOCATION,INPUT_FORMAT,serde
0,sales_avro,EXTERNAL_TABLE,s3a://format-lab/spark-hive/fmt_spark.db/sales...,org.apache.hadoop.hive.ql.io.avro.AvroContaine...,org.apache.hadoop.hive.serde2.avro.AvroSerDe
1,sales_orc,EXTERNAL_TABLE,s3a://format-lab/spark-hive/fmt_spark.db/sales...,org.apache.hadoop.hive.ql.io.orc.OrcInputFormat,org.apache.hadoop.hive.ql.io.orc.OrcSerde
2,sales_parquet,EXTERNAL_TABLE,s3a://format-lab/spark-hive/fmt_spark.db/sales...,org.apache.hadoop.hive.ql.io.parquet.MapredPar...,org.apache.hadoop.hive.ql.io.parquet.serde.Par...
3,sales_textfile,EXTERNAL_TABLE,s3a://format-lab/spark-hive/fmt_spark.db/sales...,org.apache.hadoop.mapred.TextInputFormat,org.apache.hadoop.hive.serde2.lazy.LazySimpleS...


In [12]:
# 5.3 Partitioned table: static + dynamic partition insert
# Hive syntax: partition columns are declared separately, not in the column list
spark.sql(f"DROP TABLE IF EXISTS {DB}.sales_part")
spark.sql(f"""
    CREATE TABLE {DB}.sales_part (
        id INT, big_id BIGINT, customer STRING, amount DECIMAL(10,2), sale_ts TIMESTAMP, is_paid BOOLEAN
    )
    PARTITIONED BY (sale_date DATE)
    STORED AS PARQUET
""")

# static: the partition value is written in the SQL
spark.sql(f"""
    INSERT INTO {DB}.sales_part PARTITION (sale_date = '2026-01-01')
    SELECT {COLS} FROM sales_src WHERE sale_date = '2026-01-01'
""")

# dynamic: the partition value comes from the last column of the SELECT
spark.sql("SET hive.exec.dynamic.partition.mode=nonstrict")
spark.sql(f"""
    INSERT INTO {DB}.sales_part PARTITION (sale_date)
    SELECT {COLS}, sale_date FROM sales_src WHERE sale_date > '2026-01-01'
""")

spark.sql(f"SHOW PARTITIONS {DB}.sales_part").show()
show_files(f"{SPARK_HIVE}/fmt_spark.db/sales_part")

26/11/23 05:11:49 WARN SetCommand: 'SET hive.exec.dynamic.partition.mode=nonstrict' might not work, since Spark doesn't support changing the Hive config dynamically. Please pass the Hive-specific config by adding the prefix spark.hadoop (e.g. spark.hadoop.hive.exec.dynamic.partition.mode) when starting a Spark application. For details, see the link: https://spark.apache.org/docs/latest/configuration.html#dynamically-loading-spark-properties.


+--------------------+
|           partition|
+--------------------+
|sale_date=2026-01-01|
|sale_date=2026-01-02|
|sale_date=2026-01-03|
|sale_date=2026-01-04|
|sale_date=2026-01-05|
+--------------------+

     0.0 KB  _SUCCESS
    21.1 KB  sale_date=2026-01-01/part-00000-482a64e6-7e4c-4da3-93f5-daaa9a7c0282.c000.snappy.parquet
    21.1 KB  sale_date=2026-01-01/part-00001-482a64e6-7e4c-4da3-93f5-daaa9a7c0282.c000.snappy.parquet
    23.7 KB  sale_date=2026-01-02/part-00000-c2b36171-1a7c-4ff3-a4f9-781c1e60b0e4.c000.snappy.parquet
    23.7 KB  sale_date=2026-01-02/part-00001-c2b36171-1a7c-4ff3-a4f9-781c1e60b0e4.c000.snappy.parquet
    23.7 KB  sale_date=2026-01-03/part-00000-c2b36171-1a7c-4ff3-a4f9-781c1e60b0e4.c000.snappy.parquet
    23.7 KB  sale_date=2026-01-03/part-00001-c2b36171-1a7c-4ff3-a4f9-781c1e60b0e4.c000.snappy.parquet
    23.6 KB  sale_date=2026-01-04/part-00000-c2b36171-1a7c-4ff3-a4f9-781c1e60b0e4.c000.snappy.parquet
    23.7 KB  sale_date=2026-01-04/part-00001-c2b36171-1a

In [13]:
# 5.4 Files copied in by hand are invisible until the metastore knows the partition
new_folder = f"{SPARK_HIVE}/fmt_spark.db/sales_part/sale_date=2026-02-01"
df.select("id", "big_id", "customer", "amount", "sale_ts", "is_paid").limit(100) \
  .coalesce(1).write.mode("overwrite").parquet(new_folder)

print("before repair:", spark.table(f"{DB}.sales_part").filter("sale_date = '2026-02-01'").count())

spark.sql(f"MSCK REPAIR TABLE {DB}.sales_part")        # scans the folders, adds missing partitions
# same thing for one partition: ALTER TABLE ... ADD PARTITION (sale_date='2026-02-01')

print("after repair: ", spark.table(f"{DB}.sales_part").filter("sale_date = '2026-02-01'").count())

before repair: 0
after repair:  100


In [14]:
# 5.5 INSERT OVERWRITE of one partition: only that folder is replaced, the others stay
spark.sql(f"""
    INSERT OVERWRITE TABLE {DB}.sales_part PARTITION (sale_date = '2026-01-01')
    SELECT {COLS} FROM sales_src WHERE sale_date = '2026-01-01' LIMIT 10
""")

spark.sql(f"SELECT sale_date, count(*) AS rows FROM {DB}.sales_part GROUP BY sale_date ORDER BY sale_date").show()

+----------+----+
| sale_date|rows|
+----------+----+
|2026-01-01|  10|
|2026-01-02|2000|
|2026-01-03|2000|
|2026-01-04|2000|
|2026-01-05|2000|
|2026-02-01| 100|
+----------+----+



In [15]:
# 5.6 Bucketed table: rows are hashed on `id` into 4 buckets -> bucket number in the file name
# (This is Spark's own bucketing. Hive's bucketing, with a different hash and file naming, comes in Stage 6.)
spark.sql(f"DROP TABLE IF EXISTS {DB}.sales_bucketed")
spark.sql(f"""
    CREATE TABLE {DB}.sales_bucketed
    USING PARQUET
    CLUSTERED BY (id) INTO 4 BUCKETS
    AS SELECT * FROM sales_src
""")
show_files(f"{SPARK_HIVE}/fmt_spark.db/sales_bucketed")    # part-00000-..._00000.c000.snappy.parquet ... _00003

hms("""
    SELECT t."TBL_NAME", s."NUM_BUCKETS" FROM "TBLS" t JOIN "SDS" s ON t."SD_ID" = s."SD_ID"
    WHERE t."TBL_NAME" = 'sales_bucketed'
""")

     0.0 KB  _SUCCESS
    33.8 KB  part-00000-451de7e2-3ba7-4045-92b0-9cb4cd802cd3_00000.c000.snappy.parquet
    33.9 KB  part-00000-451de7e2-3ba7-4045-92b0-9cb4cd802cd3_00001.c000.snappy.parquet
    33.7 KB  part-00000-451de7e2-3ba7-4045-92b0-9cb4cd802cd3_00002.c000.snappy.parquet
    32.7 KB  part-00000-451de7e2-3ba7-4045-92b0-9cb4cd802cd3_00003.c000.snappy.parquet
    33.1 KB  part-00001-451de7e2-3ba7-4045-92b0-9cb4cd802cd3_00000.c000.snappy.parquet
    33.6 KB  part-00001-451de7e2-3ba7-4045-92b0-9cb4cd802cd3_00001.c000.snappy.parquet
    33.6 KB  part-00001-451de7e2-3ba7-4045-92b0-9cb4cd802cd3_00002.c000.snappy.parquet
    34.0 KB  part-00001-451de7e2-3ba7-4045-92b0-9cb4cd802cd3_00003.c000.snappy.parquet
9 files


,TBL_NAME,NUM_BUCKETS
0,sales_bucketed,-1


In [16]:
# 5.7 Schema change: add a column
# Only the metastore changes. Old files don't have the column, so it reads as NULL.
spark.sql(f"ALTER TABLE {DB}.sales_parquet ADD COLUMNS (channel STRING)")
spark.sql(f"SELECT id, customer, channel FROM {DB}.sales_parquet").show(3)

# rename a column: not supported by Spark on Hive tables (we try it in Hive, Stage 6)
try:
    spark.sql(f"ALTER TABLE {DB}.sales_parquet RENAME COLUMN customer TO client")
except Exception as e:
    print("rename failed:", str(e).splitlines()[0][:150])

+---+----------+-------+
| id|  customer|channel|
+---+----------+-------+
|  0|      NULL|   NULL|
|  1|customer_1|   NULL|
|  2|customer_2|   NULL|
+---+----------+-------+
only showing top 3 rows
rename failed: [UNSUPPORTED_FEATURE.TABLE_OPERATION] The feature is not supported: Table `spark_catalog`.`fmt_spark`.`sales_parquet` does not support RENAME COLUMN. 


In [17]:
# 5.8 UPDATE / DELETE: not supported on plain Hive tables
for sql in [f"UPDATE {DB}.sales_parquet SET amount = 0 WHERE id = 1",
            f"DELETE FROM {DB}.sales_parquet WHERE id = 1"]:
    try:
        spark.sql(sql)
    except Exception as e:
        print(sql.split()[0], "failed:", str(e).splitlines()[0][:150])

UPDATE failed: [UNSUPPORTED_FEATURE.TABLE_OPERATION] The feature is not supported: Table `spark_catalog`.`fmt_spark`.`sales_parquet` does not support UPDATE TABLE. P
DELETE failed: [UNSUPPORTED_FEATURE.TABLE_OPERATION] The feature is not supported: Table `spark_catalog`.`fmt_spark`.`sales_parquet` does not support DELETE. Please 


### 5.9 Catalog: what the metastore stores for a Hive table
| HMS table | One row per | Holds |
|---|---|---|
| `DBS` | database | name, location |
| `TBLS` | table | name, type (MANAGED / EXTERNAL), owner |
| `SDS` | storage descriptor | location, input/output format, number of buckets |
| `SERDES` | serde | the class that turns bytes into rows |
| `COLUMNS_V2` | column | name, type, position |
| `PARTITIONS` | **partition** | partition name; each partition has its own `SDS` row (location) |
| `TABLE_PARAMS` | table property | stats, `EXTERNAL`, Spark's schema, ... |

In [18]:
# 5.9 (cont.) Columns of one table
hms("""
    SELECT c."INTEGER_IDX", c."COLUMN_NAME", c."TYPE_NAME"
    FROM "COLUMNS_V2" c
    JOIN "SDS" s  ON c."CD_ID" = s."CD_ID"
    JOIN "TBLS" t ON t."SD_ID" = s."SD_ID"
    WHERE t."TBL_NAME" = 'sales_parquet'
    ORDER BY c."INTEGER_IDX"
""")

,INTEGER_IDX,COLUMN_NAME,TYPE_NAME
0,0,id,int
1,1,big_id,bigint
2,2,customer,string
3,3,amount,"decimal(10,2)"
4,4,sale_date,date
5,5,sale_ts,timestamp
6,6,is_paid,boolean
7,7,channel,string


In [19]:
# 5.9 (cont.) One row per partition, each with its own location
# Thousands of partitions = thousands of rows to read before a query can plan.
hms("""
    SELECT p."PART_NAME", s."LOCATION"
    FROM "PARTITIONS" p
    JOIN "TBLS" t ON p."TBL_ID" = t."TBL_ID"
    JOIN "SDS" s  ON p."SD_ID" = s."SD_ID"
    WHERE t."TBL_NAME" = 'sales_part'
    ORDER BY p."PART_NAME"
""")

,PART_NAME,LOCATION
0,sale_date=2026-01-01,s3a://format-lab/spark-hive/fmt_spark.db/sales...
1,sale_date=2026-01-02,s3a://format-lab/spark-hive/fmt_spark.db/sales...
2,sale_date=2026-01-03,s3a://format-lab/spark-hive/fmt_spark.db/sales...
3,sale_date=2026-01-04,s3a://format-lab/spark-hive/fmt_spark.db/sales...
4,sale_date=2026-01-05,s3a://format-lab/spark-hive/fmt_spark.db/sales...
5,sale_date=2026-02-01,s3a://format-lab/spark-hive/fmt_spark.db/sales...


In [20]:
# 5.9 (cont.) Table properties: stats and the schema Spark saves for itself
hms("""
    SELECT p."PARAM_KEY", LEFT(p."PARAM_VALUE", 120) AS value
    FROM "TABLE_PARAMS" p JOIN "TBLS" t ON p."TBL_ID" = t."TBL_ID"
    WHERE t."TBL_NAME" = 'sales_parquet'
    ORDER BY 1
""")

,PARAM_KEY,value
0,EXTERNAL,TRUE
1,external.table.purge,TRUE
2,numFiles,2
3,numFilesErasureCoded,0
4,spark.sql.create.version,4.1.0
5,spark.sql.sources.schema,"{""type"":""struct"",""fields"":[{""name"":""id"",""type""..."
6,totalSize,229379
7,transient_lastDdlTime,1795412507
8,TRANSLATED_TO_EXTERNAL,TRUE


**What to notice**
- Readers now just say `spark.table("...sales_parquet")`: path, format and schema come from the metastore
- The table is still just a folder: no transactions, no history, no UPDATE/DELETE
- Partitions live as rows in Postgres; files outside the metastore are invisible until a repair

**Lesson → Stage 6:** Spark-written tables follow Spark's conventions. How does Hive itself write them?

# Stage 6: Hive tables via HiveServer2 (native Hive + Tez)
*Same table types, written by Hive's own engine.*

- The notebook sends SQL to the `hiveserver2` container with `pyhive` (FORMAT_LAB block in `requirements.txt`)
- Or in a terminal: `docker exec -it hiveserver2 beeline -u jdbc:hive2://localhost:10000/ -n hive`
- Database `fmt_hive`: external tables under `format-lab/hive/external/`, managed (ACID) under `format-lab/hive/managed/`
- Same metastore as Stage 5, so Spark and Hive see each other's tables

In [21]:
# helper: run SQL on HiveServer2. Returns a pandas DataFrame for queries, None for DDL / INSERT.
from pyhive import hive

hive_conn = hive.connect(host="hiveserver2", port=10000, username="hive")

def hive_sql(sql):
    cur = hive_conn.cursor()
    cur.execute(sql)
    if cur.description is None:                 # DDL / INSERT: nothing to return
        return None
    cols = [d[0].split(".")[-1] for d in cur.description]   # "tbl.col" -> "col"
    return pd.DataFrame(cur.fetchall(), columns=cols)

hive_sql("SELECT version()")

,_c0
0,4.0.0 r183f8cb41d3dbed961ffd27999876468ff06690c


In [ ]:
HIVE_EXT = f"{BASE}/hive/external"
HIVE_MAN = f"{BASE}/hive/managed"

hive_sql(f"""
    CREATE DATABASE IF NOT EXISTS fmt_hive
    LOCATION '{HIVE_EXT}/fmt_hive.db'
    MANAGEDLOCATION '{HIVE_MAN}/fmt_hive.db'
""")
hms('SELECT "NAME", "DB_LOCATION_URI", "DB_MANAGED_LOCATION_URI" FROM "DBS" WHERE "NAME" LIKE \'fmt_%\'')

In [ ]:
# 6.1 Recreate the Stage 5 tables with Hive: same 4 formats, external tables
# The data comes from the Spark-written table fmt_spark.sales_orc: Hive reads it through the shared metastore.
COLS_ALL = "id, big_id, customer, amount, sale_date, sale_ts, is_paid"

for fmt in ["textfile", "orc", "parquet", "avro"]:
    hive_sql(f"DROP TABLE IF EXISTS fmt_hive.sales_{fmt}")
    row_format = "ROW FORMAT DELIMITED FIELDS TERMINATED BY ','" if fmt == "textfile" else ""
    hive_sql(f"""
        CREATE EXTERNAL TABLE fmt_hive.sales_{fmt} (
            id INT, big_id BIGINT, customer STRING, amount DECIMAL(10,2),
            sale_date DATE, sale_ts TIMESTAMP, is_paid BOOLEAN
        ) {row_format} STORED AS {fmt}
    """)
    hive_sql(f"INSERT INTO fmt_hive.sales_{fmt} SELECT {COLS_ALL} FROM fmt_spark.sales_orc")

hms_tables("fmt_hive")

In [ ]:
# 6.2 File names: Hive `000000_0` vs Spark `part-00000-<uuid>.snappy.parquet`
for fmt in ["orc", "parquet"]:
    print(f"--- Hive  sales_{fmt}");  show_files(f"{HIVE_EXT}/fmt_hive.db/sales_{fmt}", limit=3)
    print(f"--- Spark sales_{fmt}");  show_files(f"{SPARK_HIVE}/fmt_spark.db/sales_{fmt}", limit=3)

In [ ]:
# 6.2 (cont.) Partitioned table written by Hive: same `sale_date=.../` folders, Hive file names
hive_sql("DROP TABLE IF EXISTS fmt_hive.sales_part")
hive_sql("""
    CREATE EXTERNAL TABLE fmt_hive.sales_part (
        id INT, big_id BIGINT, customer STRING, amount DECIMAL(10,2), sale_ts TIMESTAMP, is_paid BOOLEAN
    )
    PARTITIONED BY (sale_date DATE)
    STORED AS PARQUET
""")
hive_sql(f"INSERT INTO fmt_hive.sales_part PARTITION (sale_date) SELECT {COLS}, sale_date FROM fmt_hive.sales_orc")

show_files(f"{HIVE_EXT}/fmt_hive.db/sales_part")

In [ ]:
# 6.3 Bucketing written by Hive: one file per bucket, 000000_0 ... 000003_0
# Hive hashes with its own function; Spark's bucketing (5.6) uses a different hash,
# so the two engines can't use each other's buckets.
hive_sql("DROP TABLE IF EXISTS fmt_hive.sales_bucketed")
hive_sql("""
    CREATE EXTERNAL TABLE fmt_hive.sales_bucketed (
        id INT, big_id BIGINT, customer STRING, amount DECIMAL(10,2),
        sale_date DATE, sale_ts TIMESTAMP, is_paid BOOLEAN
    )
    CLUSTERED BY (id) INTO 4 BUCKETS
    STORED AS ORC
""")
hive_sql(f"INSERT INTO fmt_hive.sales_bucketed SELECT {COLS_ALL} FROM fmt_hive.sales_orc")

print("--- Hive bucketing");  show_files(f"{HIVE_EXT}/fmt_hive.db/sales_bucketed")
print("--- Spark bucketing"); show_files(f"{SPARK_HIVE}/fmt_spark.db/sales_bucketed")

In [ ]:
# 6.4 Cross-engine reads through the shared metastore + MinIO
# Spark reads Hive-written tables:
for fmt in ["textfile", "orc", "parquet", "avro"]:
    print(f"Spark reads fmt_hive.sales_{fmt}:", spark.table(f"spark_catalog.fmt_hive.sales_{fmt}").count())

# Hive reads Spark-written tables:
for fmt in ["textfile", "orc", "parquet", "avro"]:
    print(f"Hive  reads fmt_spark.sales_{fmt}:", hive_sql(f"SELECT count(*) AS n FROM fmt_spark.sales_{fmt}")["n"][0])

In [ ]:
# 6.5 Rename a column: ORC vs Parquet (Spark couldn't do it in 5.7)
# Hive only changes the metastore. The files still have the old name `customer`.
# A reader that matches columns BY POSITION still finds the data; one that matches BY NAME finds nothing (NULL).
for fmt in ["orc", "parquet"]:
    hive_sql(f"DROP TABLE IF EXISTS fmt_hive.rename_{fmt}")
    hive_sql(f"CREATE EXTERNAL TABLE fmt_hive.rename_{fmt} (id INT, customer STRING) STORED AS {fmt}")
    hive_sql(f"INSERT INTO fmt_hive.rename_{fmt} SELECT id, customer FROM fmt_hive.sales_orc WHERE id BETWEEN 1 AND 3")
    hive_sql(f"ALTER TABLE fmt_hive.rename_{fmt} CHANGE customer client STRING")

    print(f"--- {fmt}")
    display(hive_sql(f"SELECT id, client FROM fmt_hive.rename_{fmt}"))

In [ ]:
# 6.6 EXPLAIN in Hive: the plan is a Tez DAG (Map 1 -> Reducer 2 ...)
plan = hive_sql("""
    EXPLAIN
    SELECT sale_date, sum(amount) FROM fmt_hive.sales_orc GROUP BY sale_date
""")
print("\n".join(plan.iloc[:, 0]))

In [ ]:
# 6.6 (cont.) Same query in Spark: stages and exchanges instead of a Tez DAG
spark.sql("""
    SELECT sale_date, sum(amount) FROM spark_catalog.fmt_hive.sales_orc GROUP BY sale_date
""").explain()

In [ ]:
# 6.7 Catalog: Spark- and Hive-written tables side by side, same metastore
hms("""
    SELECT d."NAME" AS db, t."TBL_NAME", t."TBL_TYPE", s."LOCATION"
    FROM "TBLS" t
    JOIN "DBS" d ON t."DB_ID" = d."DB_ID"
    JOIN "SDS" s ON t."SD_ID" = s."SD_ID"
    WHERE d."NAME" IN ('fmt_spark', 'fmt_hive')
    ORDER BY t."TBL_NAME", d."NAME"
""")

In [ ]:
# 6.7 (cont.) Which engine created a table: Spark leaves its own properties, Hive doesn't
hms("""
    SELECT d."NAME" AS db, t."TBL_NAME", p."PARAM_KEY", LEFT(p."PARAM_VALUE", 60) AS value
    FROM "TABLE_PARAMS" p
    JOIN "TBLS" t ON p."TBL_ID" = t."TBL_ID"
    JOIN "DBS" d  ON t."DB_ID" = d."DB_ID"
    WHERE t."TBL_NAME" = 'sales_orc' AND d."NAME" IN ('fmt_spark', 'fmt_hive')
    ORDER BY 1, 3
""")

**What to notice**
- Same metastore, same storage: both engines read each other's plain tables
- File names, bucketing hash and column matching (position vs name) differ per engine

**Lesson → Stage 7:** still no updates/deletes, no atomic multi-file commits, and readers can see half-written data.

# Stage 7: Hive ACID
*Fixes: updates and deletes. Each transaction gets its own folder.*

Managed ORC table with `transactional=true`, in `format-lab/hive/managed/fmt_hive.db/`.
The transaction manager (`DbTxnManager`) and the compactor are turned on in `hive_custom_conf/hive-site.xml`.

In [ ]:
ACID = f"{HIVE_MAN}/fmt_hive.db/sales_acid"

hive_sql("DROP TABLE IF EXISTS fmt_hive.sales_acid")
hive_sql("""
    CREATE TABLE fmt_hive.sales_acid (
        id INT, customer STRING, amount DECIMAL(10,2), sale_date DATE
    )
    STORED AS ORC
    TBLPROPERTIES ('transactional' = 'true')
""")
hive_sql("DESCRIBE FORMATTED fmt_hive.sales_acid").tail(20)

In [ ]:
# 7.1 INSERT: every transaction writes a new folder delta_<writeId>_<writeId>_<stmtId>/
hive_sql("INSERT INTO fmt_hive.sales_acid SELECT id, customer, amount, sale_date FROM fmt_hive.sales_orc WHERE id < 100")
hive_sql("INSERT INTO fmt_hive.sales_acid SELECT id, customer, amount, sale_date FROM fmt_hive.sales_orc WHERE id BETWEEN 100 AND 199")

show_files(ACID)        # delta_0000001_0000001_0000/bucket_00000, delta_0000002_...

In [ ]:
# 7.1 (cont.) Inside an ACID ORC file: Hive adds hidden columns to every row
# operation, originalTransaction, bucket, rowId, currentTransaction, then the real row as a struct
local = download_first(f"{ACID}/delta_0000001_0000001_0000", "bucket_00000", "/tmp/acid_bucket.orc")
print(po.ORCFile(local).schema)
po.ORCFile(local).read().to_pandas().head(3)

In [ ]:
# 7.2 UPDATE = delete + insert -> a delete_delta_ folder and a delta_ folder
hive_sql("UPDATE fmt_hive.sales_acid SET amount = 0 WHERE id < 10")
show_files(ACID)

In [ ]:
# 7.3 DELETE -> only a delete_delta_ folder (row ids of the deleted rows)
hive_sql("DELETE FROM fmt_hive.sales_acid WHERE id BETWEEN 10 AND 19")
show_files(ACID)

hive_sql("SELECT count(*) AS rows, sum(CASE WHEN amount = 0 THEN 1 ELSE 0 END) AS zero_amount FROM fmt_hive.sales_acid")

In [ ]:
# 7.3 (cont.) MERGE INTO: update matching rows, insert new ones, in one transaction
hive_sql("""
    MERGE INTO fmt_hive.sales_acid AS t
    USING (SELECT id, customer, amount, sale_date FROM fmt_hive.sales_orc WHERE id BETWEEN 190 AND 209) AS s
    ON t.id = s.id
    WHEN MATCHED THEN UPDATE SET amount = 999
    WHEN NOT MATCHED THEN INSERT VALUES (s.id, s.customer, s.amount, s.sale_date)
""")
show_files(ACID)

hive_sql("SELECT count(*) AS rows FROM fmt_hive.sales_acid")

In [ ]:
# 7.4 Compaction
# minor: merges many delta_ folders into one delta_ (and delete_delta_ into one delete_delta_)
# major: rewrites everything into base_<writeId>/, deleted rows are gone for good
hive_sql("ALTER TABLE fmt_hive.sales_acid COMPACT 'minor'")
hive_sql("SHOW COMPACTIONS")

In [ ]:
# 7.4 (cont.) Compaction runs in the background. Run this cell again until state = succeeded / ready for cleaning
display(hive_sql("SHOW COMPACTIONS"))
show_files(ACID)

In [ ]:
# 7.4 (cont.) Major compaction -> base_N/
hive_sql("ALTER TABLE fmt_hive.sales_acid COMPACT 'major'")
display(hive_sql("SHOW COMPACTIONS"))

In [ ]:
# 7.4 (cont.) Run again until done: one base_ folder; old delta_ folders go after the cleaner runs
display(hive_sql("SHOW COMPACTIONS"))
show_files(ACID)

In [ ]:
# 7.5 Insert-only ACID with Parquet: transactions for INSERT, but no UPDATE / DELETE
hive_sql("DROP TABLE IF EXISTS fmt_hive.sales_insert_only")
hive_sql("""
    CREATE TABLE fmt_hive.sales_insert_only (id INT, customer STRING, amount DECIMAL(10,2))
    STORED AS PARQUET
    TBLPROPERTIES ('transactional' = 'true', 'transactional_properties' = 'insert_only')
""")
hive_sql("INSERT INTO fmt_hive.sales_insert_only SELECT id, customer, amount FROM fmt_hive.sales_orc WHERE id < 50")
hive_sql("INSERT INTO fmt_hive.sales_insert_only SELECT id, customer, amount FROM fmt_hive.sales_orc WHERE id BETWEEN 50 AND 99")

show_files(f"{HIVE_MAN}/fmt_hive.db/sales_insert_only")     # delta_ folders with plain Parquet files

try:
    hive_sql("UPDATE fmt_hive.sales_insert_only SET amount = 0 WHERE id = 1")
except Exception as e:
    print("update failed:", str(e)[:200])

In [ ]:
# 7.6 Read the ACID table from Spark
# Spark doesn't understand Hive's delta_ / delete_delta_ folders: it fails or returns wrong rows.
print("Hive count:", hive_sql("SELECT count(*) AS n FROM fmt_hive.sales_acid")["n"][0])
try:
    acid_spark = spark.table("spark_catalog.fmt_hive.sales_acid")
    print("Spark count:", acid_spark.count())
    acid_spark.show(3)
except Exception as e:
    print("Spark read failed:", str(e).splitlines()[0][:200])

### 7.7 Catalog: the transaction state lives in the metastore database
| HMS table | Holds |
|---|---|
| `TXNS` | every transaction: id, state (open / committed / aborted), start time |
| `TXN_TO_WRITE_ID` | which transaction got which write id for which table |
| `NEXT_WRITE_ID` | next write id per table (the numbers in `delta_N_N`) |
| `COMPACTION_QUEUE` | compactions waiting / running |
| `COMPLETED_COMPACTIONS` | compaction history |
| `HIVE_LOCKS` | locks held right now |

In [ ]:
hms('SELECT * FROM "TXNS" ORDER BY "TXN_ID" DESC LIMIT 10')

In [ ]:
hms("""SELECT * FROM "TXN_TO_WRITE_ID" WHERE "T2W_DATABASE" = 'fmt_hive' ORDER BY "T2W_WRITEID" """)

In [ ]:
hms("""SELECT * FROM "NEXT_WRITE_ID" WHERE "NWI_DATABASE" = 'fmt_hive'""")

In [ ]:
display(hms('SELECT * FROM "COMPACTION_QUEUE"'))
hms('SELECT * FROM "COMPLETED_COMPACTIONS" ORDER BY 1 DESC LIMIT 10')

**What to notice**
- Writes never change old files: new `delta_` / `delete_delta_` folders, merged later by compaction
- A reader must combine base + deltas − delete deltas, using the open/committed transactions from the metastore DB
- Only Hive knows how to do that; Spark can't read the table correctly

**Lesson → Stage 8:** ACID works only through Hive's engine, the transaction state sits in a central DB,
there's no time travel, and partition listing gets slow at scale.

# Stage 8: Delta Lake: skipped
Delta and Iceberg solve the same problems in a similar way (metadata files next to the data).
The lakehouse already runs Iceberg, so the lab goes straight to Stage 9.

# Stage 9: Apache Iceberg (Spark)
*Fixes: partition lock-in and the single-engine design. A metadata tree with hidden partitioning, built to be engine-neutral.*

- Catalog `lakehouse` (Iceberg on the Hive Metastore), namespace `fmt_iceberg`, files under `format-lab/iceberg/`
- Iceberg writes with its own S3 client, so table locations use `s3://`.
  The notebook helpers use Hadoop (`s3a://`), so they convert the path first. Same bucket, same files.

```
catalog (HMS)        TBLS + TABLE_PARAMS: metadata_location ──┐
                                                               ▼
metadata/00003-<uuid>.metadata.json      table metadata: schemas, partition specs, snapshots, current snapshot
        │  current snapshot -> manifest list
        ▼
metadata/snap-<id>-<uuid>.avro           manifest list: one row per manifest (+ partition summary)
        │
        ▼
metadata/<uuid>-m0.avro                  manifest: one row per data/delete file (+ partition, row count, min/max)
        │
        ▼
data/...parquet                          data files (and delete files)
```

In [22]:
ICE    = "lakehouse.fmt_iceberg"
ICE_S3 = "s3://format-lab/iceberg"          # for Iceberg (S3FileIO)
ICE_A  = f"{BASE}/iceberg"                  # same place, for the Hadoop helpers

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {ICE} LOCATION '{ICE_S3}/fmt_iceberg.db'")
df.createOrReplaceTempView("sales_src")

In [23]:
# helpers to look at Iceberg files
import json, fastavro

def to_s3a(p):
    return p.replace("s3://", "s3a://", 1)

def download(path, local):
    """Copy one exact file from MinIO to the container."""
    jvm = spark._jvm
    p = jvm.org.apache.hadoop.fs.Path(to_s3a(path))
    p.getFileSystem(spark._jsc.hadoopConfiguration()).copyToLocalFile(p, jvm.org.apache.hadoop.fs.Path(local))
    return local

def read_json_file(path):
    with open(download(path, "/tmp/iceberg.metadata.json")) as f:
        return json.load(f)

def read_avro_file(path):
    with open(download(path, "/tmp/iceberg_file.avro"), "rb") as f:
        return list(fastavro.reader(f))

def table_location(table):
    rows = spark.sql(f"DESCRIBE TABLE EXTENDED {table}").filter("col_name = 'Location'").collect()
    return to_s3a(rows[0]["data_type"])

def current_metadata_file(table):
    return spark.sql(f"SELECT file FROM {table}.metadata_log_entries ORDER BY timestamp DESC LIMIT 1").first()[0]

In [24]:
# 9.1 Create + insert
spark.sql(f"DROP TABLE IF EXISTS {ICE}.sales PURGE")
spark.sql(f"""
    CREATE TABLE {ICE}.sales (
        id INT, big_id BIGINT, customer STRING, amount DECIMAL(10,2),
        sale_date DATE, sale_ts TIMESTAMP, is_paid BOOLEAN
    )
    USING iceberg
    TBLPROPERTIES ('format-version' = '2')
""")
print("after CREATE:"); show_files(table_location(f"{ICE}.sales"))     # only a metadata.json, no snapshot yet

spark.sql(f"INSERT INTO {ICE}.sales SELECT * FROM sales_src")
print("\nafter INSERT:"); show_files(table_location(f"{ICE}.sales"))  # + data file, manifest list, manifest, new metadata.json

after CREATE:
     1.0 KB  metadata/00000-3d0a2c15-a410-435c-a9d6-709305adb9bc.metadata.json
1 files



after INSERT:
    56.5 KB  data/00000-38-051ce8a2-a699-4d13-962e-8119c7e290a0-0-00001.parquet
    56.1 KB  data/00001-39-051ce8a2-a699-4d13-962e-8119c7e290a0-0-00001.parquet
     1.0 KB  metadata/00000-3d0a2c15-a410-435c-a9d6-709305adb9bc.metadata.json
     2.2 KB  metadata/00001-0f4b0000-5bca-40b8-925c-84e01a9b837f.metadata.json
     7.4 KB  metadata/32ef7f28-adf5-48fe-905a-1b67c29331a6-m0.avro
     4.4 KB  metadata/snap-1347610931031077484-1-32ef7f28-adf5-48fe-905a-1b67c29331a6.avro
6 files


In [25]:
# 9.1 (cont.) Level 1: the table metadata file (JSON)
# HiveCatalog names it 00001-<uuid>.metadata.json (a Hadoop catalog would use v1.metadata.json)
meta_path = current_metadata_file(f"{ICE}.sales")
meta = read_json_file(meta_path)
print(meta_path, "\n")

print("format-version:     ", meta["format-version"])
print("location:           ", meta["location"])
print("current-snapshot-id:", meta["current-snapshot-id"])
print("schema (with field ids):")
print(json.dumps(meta["schemas"][-1], indent=1)[:800])
print("partition-specs:    ", meta["partition-specs"])
print("snapshots:")
for s in meta["snapshots"]:
    print("  ", s["snapshot-id"], s["summary"]["operation"], "->", s["manifest-list"])

s3://format-lab/iceberg/fmt_iceberg.db/sales/metadata/00001-0f4b0000-5bca-40b8-925c-84e01a9b837f.metadata.json 

format-version:      2
location:            s3://format-lab/iceberg/fmt_iceberg.db/sales
current-snapshot-id: 1347610931031077484
schema (with field ids):
{
 "type": "struct",
 "schema-id": 0,
 "fields": [
  {
   "id": 1,
   "name": "id",
   "required": false,
   "type": "int"
  },
  {
   "id": 2,
   "name": "big_id",
   "required": false,
   "type": "long"
  },
  {
   "id": 3,
   "name": "customer",
   "required": false,
   "type": "string"
  },
  {
   "id": 4,
   "name": "amount",
   "required": false,
   "type": "decimal(10, 2)"
  },
  {
   "id": 5,
   "name": "sale_date",
   "required": false,
   "type": "date"
  },
  {
   "id": 6,
   "name": "sale_ts",
   "required": false,
   "type": "timestamptz"
  },
  {
   "id": 7,
   "name": "is_paid",
   "required": false,
   "type": "boolean"
  }
 ]
}
partition-specs:     [{'spec-id': 0, 'fields': []}]
snapshots:
   1347610931031

In [26]:
# 9.1 (cont.) Level 2: the manifest list (Avro), one row per manifest
snapshot = [s for s in meta["snapshots"] if s["snapshot-id"] == meta["current-snapshot-id"]][0]
manifest_list = read_avro_file(snapshot["manifest-list"])

for m in manifest_list:
    print({k: m.get(k) for k in ["manifest_path", "manifest_length", "partition_spec_id", "content",
                                 "added_snapshot_id", "added_files_count", "added_rows_count"]})

{'manifest_path': 's3://format-lab/iceberg/fmt_iceberg.db/sales/metadata/32ef7f28-adf5-48fe-905a-1b67c29331a6-m0.avro', 'manifest_length': 7533, 'partition_spec_id': 0, 'content': 0, 'added_snapshot_id': 1347610931031077484, 'added_files_count': 2, 'added_rows_count': 10000}


In [27]:
# 9.1 (cont.) Level 3: the manifest (Avro), one row per data file, with stats per column (by field id)
entries = read_avro_file(manifest_list[0]["manifest_path"])

for e in entries:
    f = e["data_file"]
    print("status:      ", e["status"], "(1 = added, 0 = existing, 2 = deleted)")
    print("file:        ", f["file_path"])
    print("format:      ", f["file_format"], "  rows:", f["record_count"], "  bytes:", f["file_size_in_bytes"])
    print("partition:   ", f["partition"])
    print("null counts: ", f["null_value_counts"])
    print("lower bounds:", f["lower_bounds"][:3], "...")      # field id -> min value (binary)

status:       1 (1 = added, 0 = existing, 2 = deleted)
file:         s3://format-lab/iceberg/fmt_iceberg.db/sales/data/00000-38-051ce8a2-a699-4d13-962e-8119c7e290a0-0-00001.parquet
format:       PARQUET   rows: 5000   bytes: 57836
partition:    {}
null counts:  [{'key': 1, 'value': 0}, {'key': 2, 'value': 0}, {'key': 3, 'value': 500}, {'key': 4, 'value': 500}, {'key': 5, 'value': 0}, {'key': 6, 'value': 0}, {'key': 7, 'value': 0}]
lower bounds: [{'key': 1, 'value': b'\x00\x00\x00\x00'}, {'key': 2, 'value': b'\x00\x00\x00\x00\x00\x00\x00\x00'}, {'key': 3, 'value': b'customer_1'}] ...
status:       1 (1 = added, 0 = existing, 2 = deleted)
file:         s3://format-lab/iceberg/fmt_iceberg.db/sales/data/00001-39-051ce8a2-a699-4d13-962e-8119c7e290a0-0-00001.parquet
format:       PARQUET   rows: 5000   bytes: 57486
partition:    {}
null counts:  [{'key': 1, 'value': 0}, {'key': 2, 'value': 0}, {'key': 3, 'value': 500}, {'key': 4, 'value': 500}, {'key': 5, 'value': 0}, {'key': 6, 'value': 0},

In [28]:
# 9.1 (cont.) A second insert: a new snapshot, a new manifest list, a new metadata.json.
# Old files stay; the new snapshot points at the old manifest + a new one.
spark.sql(f"INSERT INTO {ICE}.sales SELECT * FROM sales_src LIMIT 100")

meta = read_json_file(current_metadata_file(f"{ICE}.sales"))
snapshot = [s for s in meta["snapshots"] if s["snapshot-id"] == meta["current-snapshot-id"]][0]
for m in read_avro_file(snapshot["manifest-list"]):
    print(m["manifest_path"].split("/")[-1], " added by snapshot", m["added_snapshot_id"])

show_files(table_location(f"{ICE}.sales"))

0d0817a7-6c75-48b7-a361-611a8cde2771-m0.avro  added by snapshot 5281010887262651805
32ef7f28-adf5-48fe-905a-1b67c29331a6-m0.avro  added by snapshot 1347610931031077484
    56.5 KB  data/00000-38-051ce8a2-a699-4d13-962e-8119c7e290a0-0-00001.parquet
     3.2 KB  data/00000-45-350d2ea1-887f-45ed-b79b-5bb16ff90602-0-00001.parquet
    56.1 KB  data/00001-39-051ce8a2-a699-4d13-962e-8119c7e290a0-0-00001.parquet
     1.0 KB  metadata/00000-3d0a2c15-a410-435c-a9d6-709305adb9bc.metadata.json
     2.2 KB  metadata/00001-0f4b0000-5bca-40b8-925c-84e01a9b837f.metadata.json
     3.3 KB  metadata/00002-de2a8a42-5132-4eb1-a022-160dc3fdfafe.metadata.json
     7.3 KB  metadata/0d0817a7-6c75-48b7-a361-611a8cde2771-m0.avro
     7.4 KB  metadata/32ef7f28-adf5-48fe-905a-1b67c29331a6-m0.avro
     4.4 KB  metadata/snap-1347610931031077484-1-32ef7f28-adf5-48fe-905a-1b67c29331a6.avro
     4.4 KB  metadata/snap-5281010887262651805-1-0d0817a7-6c75-48b7-a361-611a8cde2771.avro
10 files


In [ ]:
# 9.2 Data file formats: Parquet / ORC / Avro, chosen per table with write.format.default
for fmt in ["parquet", "orc", "avro"]:
    spark.sql(f"DROP TABLE IF EXISTS {ICE}.sales_{fmt} PURGE")
    spark.sql(f"""
        CREATE TABLE {ICE}.sales_{fmt} USING iceberg
        TBLPROPERTIES ('write.format.default' = '{fmt}')
        AS SELECT * FROM sales_src
    """)
    print(f"--- {fmt}")
    show_files(table_location(f"{ICE}.sales_{fmt}") + "/data")

# same metadata tree for all three; only the data files differ
spark.sql(f"SELECT file_format, count(*) AS files, sum(record_count) AS rows FROM {ICE}.sales_orc.files GROUP BY 1").show()

In [ ]:
# 9.3 Hidden partitioning: partition by a TRANSFORM of a column, not by an extra column
# days(sale_ts) -> folder sale_ts_day=2026-01-01 ; bucket(4, id) -> folder id_bucket=0..3
spark.sql(f"DROP TABLE IF EXISTS {ICE}.sales_hidden PURGE")
spark.sql(f"""
    CREATE TABLE {ICE}.sales_hidden (
        id INT, big_id BIGINT, customer STRING, amount DECIMAL(10,2),
        sale_date DATE, sale_ts TIMESTAMP, is_paid BOOLEAN
    )
    USING iceberg
    PARTITIONED BY (days(sale_ts), bucket(4, id))
""")
spark.sql(f"INSERT INTO {ICE}.sales_hidden SELECT * FROM sales_src")

show_files(table_location(f"{ICE}.sales_hidden") + "/data", limit=10)

In [ ]:
# 9.3 (cont.) Query with a normal filter on sale_ts: Iceberg turns it into a partition filter by itself.
# In Stage 4/5 you had to filter on the partition column (sale_date) to get pruning.
q = spark.sql(f"""
    SELECT count(*) FROM {ICE}.sales_hidden
    WHERE sale_ts >= TIMESTAMP '2026-01-03 00:00:00' AND sale_ts < TIMESTAMP '2026-01-04 00:00:00'
""")
q.explain()          # the filter is pushed into the Iceberg scan
q.show()

# Spark UI (http://localhost:4040 -> SQL tab -> the scan): compare "number of result files" with the total
spark.sql(f"SELECT count(*) AS total_data_files FROM {ICE}.sales_hidden.files").show()

In [ ]:
# 9.3 (cont.) Partition info per data file (from the manifests)
spark.sql(f"SELECT partition, record_count, file_path FROM {ICE}.sales_hidden.files ORDER BY partition").show(8, truncate=60)

In [ ]:
# 9.4 Partition evolution: change the spec WITHOUT rewriting old files
# Old files keep spec 0 (days + bucket), new files use spec 1 (days only)
spark.sql(f"ALTER TABLE {ICE}.sales_hidden DROP PARTITION FIELD bucket(4, id)")
spark.sql(f"INSERT INTO {ICE}.sales_hidden SELECT * FROM sales_src LIMIT 500")

spark.sql(f"""
    SELECT spec_id, partition, count(*) AS files, sum(record_count) AS rows
    FROM {ICE}.sales_hidden.files GROUP BY spec_id, partition ORDER BY spec_id, partition
""").show(truncate=False)

meta = read_json_file(current_metadata_file(f"{ICE}.sales_hidden"))
print("default-spec-id:", meta["default-spec-id"])
for spec in meta["partition-specs"]:
    print(spec)

In [ ]:
# 9.4 (cont.) Folder layout: old folders sale_ts_day=.../id_bucket=N/, new ones only sale_ts_day=.../
show_files(table_location(f"{ICE}.sales_hidden") + "/data", limit=40)

In [ ]:
# 9.5 Schema evolution by column ID: no data files are rewritten
# Compare with Stage 5.7 / 6.5, where Hive matched columns by name or by position.
t = f"{ICE}.sales"
files_before = spark.sql(f"SELECT count(*) FROM {t}.files").first()[0]

spark.sql(f"ALTER TABLE {t} ADD COLUMN channel STRING")
spark.sql(f"ALTER TABLE {t} RENAME COLUMN customer TO client")
spark.sql(f"ALTER TABLE {t} DROP COLUMN big_id")
spark.sql(f"ALTER TABLE {t} ALTER COLUMN amount FIRST")            # reorder
spark.sql(f"ALTER TABLE {t} ALTER COLUMN amount TYPE DECIMAL(12,2)")  # widen a type

spark.table(t).show(3)     # renamed column still has its data, new column is NULL for old rows
print("data files before:", files_before, " after:", spark.sql(f"SELECT count(*) FROM {t}.files").first()[0])

In [ ]:
# 9.5 (cont.) Each schema version is kept in the metadata, with stable field ids.
# `client` keeps the id `customer` had, so old files (which store field ids) still match.
meta = read_json_file(current_metadata_file(f"{ICE}.sales"))
print("current-schema-id:", meta["current-schema-id"])
for schema in meta["schemas"]:
    print(schema["schema-id"], [(f["id"], f["name"]) for f in schema["fields"]])

In [ ]:
# 9.5 (cont.) The field id is written inside the Parquet file too (field_id in the Parquet schema)
data_file = spark.sql(f"SELECT file_path FROM {ICE}.sales.files LIMIT 1").first()[0]
local = download(data_file, "/tmp/iceberg_data.parquet")
print(pq.ParquetFile(local).schema)       # look for field_id=1, 2, 3 ...

In [ ]:
# 9.6 Copy-on-write vs merge-on-read for DELETE / UPDATE
# copy-on-write: rewrite every data file that has a deleted row      (slow write, fast read)
# merge-on-read: write a small delete file listing the deleted rows  (fast write, reader merges)
for mode in ["copy-on-write", "merge-on-read"]:
    name = f"{ICE}.sales_" + mode.replace("-", "_")
    spark.sql(f"DROP TABLE IF EXISTS {name} PURGE")
    spark.sql(f"""
        CREATE TABLE {name} USING iceberg
        TBLPROPERTIES ('format-version' = '2',
                       'write.delete.mode' = '{mode}',
                       'write.update.mode' = '{mode}',
                       'write.merge.mode'  = '{mode}')
        AS SELECT * FROM sales_src
    """)
    spark.sql(f"DELETE FROM {name} WHERE id < 10")

    print(f"--- {mode}")
    # content: 0 = data file, 1 = position delete file, 2 = equality delete file
    spark.sql(f"""
        SELECT content, file_format, record_count, split(file_path, '/')[size(split(file_path, '/')) - 1] AS file
        FROM {name}.all_files
    """).show(truncate=False)

In [ ]:
# 9.6 (cont.) The position delete file: which row positions of which data file are deleted
del_file = spark.sql(f"SELECT file_path FROM {ICE}.sales_merge_on_read.delete_files LIMIT 1").first()[0]
local = download(del_file, "/tmp/pos_delete.parquet")
pq.read_table(local).to_pandas().head(10)            # file_path + pos

In [ ]:
# 9.7 Format v3: merge-on-read deletes become DELETION VECTORS in a Puffin file (.puffin),
# one bitmap per data file instead of a Parquet file with (file_path, pos) rows
spark.sql(f"DROP TABLE IF EXISTS {ICE}.sales_v3 PURGE")
spark.sql(f"""
    CREATE TABLE {ICE}.sales_v3 USING iceberg
    TBLPROPERTIES ('format-version' = '3', 'write.delete.mode' = 'merge-on-read')
    AS SELECT * FROM sales_src
""")
spark.sql(f"DELETE FROM {ICE}.sales_v3 WHERE id < 10")

spark.sql(f"""
    SELECT content, file_format, record_count, split(file_path, '/')[size(split(file_path, '/')) - 1] AS file
    FROM {ICE}.sales_v3.files
""").show(truncate=False)

meta = read_json_file(current_metadata_file(f"{ICE}.sales_v3"))
print("format-version:", meta["format-version"], "  next-row-id:", meta.get("next-row-id"))   # v3 row lineage

In [ ]:
# 9.7 (cont.) v2 vs v3 in short
# v3 adds: deletion vectors, row lineage (_row_id, _last_updated_sequence_number), new types (variant, timestamp_ns, geo),
#          default values for new columns
try:
    spark.sql(f"SELECT id, _row_id, _last_updated_sequence_number FROM {ICE}.sales_v3 LIMIT 5").show()
except Exception as e:
    print("row lineage columns not available:", str(e).splitlines()[0][:150])

In [ ]:
# 9.8 Snapshots and time travel: every commit is a snapshot you can query
spark.sql(f"SELECT committed_at, snapshot_id, operation, summary['added-records'] AS added FROM {ICE}.sales.snapshots").show(truncate=False)

first_snapshot = spark.sql(f"SELECT snapshot_id FROM {ICE}.sales.snapshots ORDER BY committed_at LIMIT 1").first()[0]
first_time     = spark.sql(f"SELECT committed_at FROM {ICE}.sales.snapshots ORDER BY committed_at LIMIT 1").first()[0]

print("now:              ", spark.table(f"{ICE}.sales").count(), "rows")
print("VERSION AS OF:    ", spark.sql(f"SELECT count(*) FROM {ICE}.sales VERSION AS OF {first_snapshot}").first()[0], "rows")
print("TIMESTAMP AS OF:  ", spark.sql(f"SELECT count(*) FROM {ICE}.sales TIMESTAMP AS OF '{first_time}'").first()[0], "rows")

# old schema too: the first snapshot still has `customer` and `big_id`
spark.sql(f"SELECT * FROM {ICE}.sales VERSION AS OF {first_snapshot} LIMIT 2").show()

In [ ]:
# 9.8 (cont.) Tags and branches: named pointers to snapshots (like git)
spark.sql(f"ALTER TABLE {ICE}.sales CREATE TAG IF NOT EXISTS first_load AS OF VERSION {first_snapshot}")
spark.sql(f"ALTER TABLE {ICE}.sales CREATE BRANCH IF NOT EXISTS dev")

# write only to the branch: main doesn't change
spark.sql(f"DELETE FROM {ICE}.sales.branch_dev WHERE id < 5000")

print("main:      ", spark.table(f"{ICE}.sales").count())
print("branch dev:", spark.sql(f"SELECT count(*) FROM {ICE}.sales VERSION AS OF 'dev'").first()[0])
print("tag:       ", spark.sql(f"SELECT count(*) FROM {ICE}.sales VERSION AS OF 'first_load'").first()[0])

spark.sql(f"SELECT * FROM {ICE}.sales.refs").show(truncate=False)

In [ ]:
# 9.9 Metadata tables: the metadata tree as SQL tables
t = f"{ICE}.sales_hidden"
for mt in ["snapshots", "history", "manifests", "partitions", "files"]:
    print(f"--- {mt}")
    spark.table(f"{t}.{mt}").show(5, truncate=50)

In [ ]:
# 9.10 Maintenance 1: compact small files (rewrite_data_files)
# Make small files first: 20 tiny inserts
spark.sql(f"DROP TABLE IF EXISTS {ICE}.sales_small PURGE")
spark.sql(f"CREATE TABLE {ICE}.sales_small USING iceberg AS SELECT * FROM sales_src LIMIT 0")
for i in range(20):
    spark.sql(f"INSERT INTO {ICE}.sales_small SELECT * FROM sales_src WHERE id % 20 = {i}")

print("data files before:", spark.sql(f"SELECT count(*) FROM {ICE}.sales_small.files").first()[0])
spark.sql(f"CALL lakehouse.system.rewrite_data_files(table => 'fmt_iceberg.sales_small', options => map('min-input-files', '2'))").show()
print("data files after: ", spark.sql(f"SELECT count(*) FROM {ICE}.sales_small.files").first()[0])
# the old small files are still in MinIO: older snapshots point at them

In [ ]:
# 9.10 Maintenance 2: merge manifests (rewrite_manifests)
print("manifests before:", spark.sql(f"SELECT count(*) FROM {ICE}.sales_small.manifests").first()[0])
spark.sql("CALL lakehouse.system.rewrite_manifests('fmt_iceberg.sales_small')").show()
print("manifests after: ", spark.sql(f"SELECT count(*) FROM {ICE}.sales_small.manifests").first()[0])

In [ ]:
# 9.10 Maintenance 3: expire old snapshots -> files no snapshot needs are deleted, time travel to them is gone
loc = table_location(f"{ICE}.sales_small")
print("files in MinIO before:", len(ls_files(loc)))
print("snapshots before:     ", spark.sql(f"SELECT count(*) FROM {ICE}.sales_small.snapshots").first()[0])

spark.sql(f"""
    CALL lakehouse.system.expire_snapshots(
        table => 'fmt_iceberg.sales_small',
        older_than => current_timestamp(),
        retain_last => 1)
""").show()

print("files in MinIO after: ", len(ls_files(loc)))
print("snapshots after:      ", spark.sql(f"SELECT count(*) FROM {ICE}.sales_small.snapshots").first()[0])

In [ ]:
# 9.10 Maintenance 4: remove orphan files = files in the folder that NO metadata points at
# (e.g. left by a failed write). Put one there by hand, then dry-run the cleanup.
df.limit(5).coalesce(1).write.mode("overwrite").parquet(f"{loc}/data/orphan_test")

spark.sql(f"""
    CALL lakehouse.system.remove_orphan_files(
        table => 'fmt_iceberg.sales_small',
        older_than => current_timestamp(),
        dry_run => true)
""").show(truncate=False)
# dry_run => false would delete them. Default older_than is 3 days ago, to protect writes still running.

In [ ]:
# 9.11 Catalog: for an Iceberg table, HMS holds almost nothing, just a POINTER to the current metadata file
hms("""
    SELECT t."TBL_NAME", t."TBL_TYPE", p."PARAM_KEY", p."PARAM_VALUE"
    FROM "TABLE_PARAMS" p
    JOIN "TBLS" t ON p."TBL_ID" = t."TBL_ID"
    JOIN "DBS" d  ON t."DB_ID" = d."DB_ID"
    WHERE d."NAME" = 'fmt_iceberg' AND t."TBL_NAME" = 'sales'
      AND p."PARAM_KEY" IN ('table_type', 'metadata_location', 'previous_metadata_location')
""")

In [ ]:
# 9.11 (cont.) Each commit = write a new metadata.json, then swap the pointer in HMS (atomic, with a lock)
def pointer():
    return hms("""
        SELECT p."PARAM_VALUE" FROM "TABLE_PARAMS" p JOIN "TBLS" t ON p."TBL_ID" = t."TBL_ID"
        JOIN "DBS" d ON t."DB_ID" = d."DB_ID"
        WHERE d."NAME" = 'fmt_iceberg' AND t."TBL_NAME" = 'sales' AND p."PARAM_KEY" = 'metadata_location'
    """).iloc[0, 0].split("/")[-1]

print("before:", pointer())
spark.sql(f"INSERT INTO {ICE}.sales (id, amount, client) VALUES (99999, 1.00, 'pointer_test')")
print("after: ", pointer())

In [ ]:
# 9.11 (cont.) Compare: a Hive table (Stage 5) has its columns, partitions and SerDe in HMS;
# an Iceberg table has a placeholder SerDe and no partitions in HMS: all of that is in the metadata files.
display(hms_tables("fmt_iceberg"))
hms("""
    SELECT d."NAME" AS db, t."TBL_NAME", count(p."PART_ID") AS partitions_in_hms
    FROM "TBLS" t JOIN "DBS" d ON t."DB_ID" = d."DB_ID"
    LEFT JOIN "PARTITIONS" p ON p."TBL_ID" = t."TBL_ID"
    WHERE (d."NAME" = 'fmt_spark' AND t."TBL_NAME" = 'sales_part')
       OR (d."NAME" = 'fmt_iceberg' AND t."TBL_NAME" = 'sales_hidden')
    GROUP BY 1, 2
""")

**What to notice**
- Every change writes new files (data, manifests, manifest list, metadata.json); nothing is edited in place
- The catalog only stores one pointer; schema, partitions, snapshots and stats live in the metadata files
- Columns are matched by **field id**, so rename/drop/reorder are safe without rewriting data
- Partitioning is hidden (transforms) and can change over time without rewriting old files
- Old snapshots give time travel until `expire_snapshots` removes them

**Lesson → Stage 10:** an open format is only useful if other engines read and write it the same way.